# Tool 3bis: Automatic sleep scoring (GSSC, YASA, SleepFM)

Scores every 30-s epoch of a night automatically with up to **three published models**, in two steps:

1. **Automatic scoring** (Sections 1-3). Each model you tick scores each night. Its hypnogram is written beside
   the recording (tool-3 format, so tools 5-9 can use it in place of an expert one), with the **probability of each
   stage** for every epoch (the **hypnodensity**). Where an expert hypnogram exists, each model is compared with
   it night by night (confusion matrix, accuracy, Cohen's kappa, MCC, macro-F1, balanced accuracy, F1 per stage).
2. **Comparison of the models** (Section 4). The models side by side against the expert hypnogram, on every
   night together: agreement table, confusion matrices, hypnograms, hypnodensities, and two **recalibrations of
   SleepFM** learnt on your own expert hypnograms.

A night where the probabilities are clear-cut is easy to score; a night with many uncertain epochs is ambiguous
(for the machine and often for the scorer too).

**Agreement with the expert**: accuracy, Cohen's kappa, MCC, F1 per stage, **macro-F1** (mean of the F1 over the
stages the expert scored on that night) and **balanced accuracy** (mean, over the same stages, of the share of the
expert's epochs the model finds), per night and with every night pooled. A night where the expert scored a **single
stage** (a nap where the participant stays awake) has a kappa and an MCC of 0 whatever the model, by construction: it
is flagged, and the option under the expert hypnogram suffix (Section 1) leaves it out of the kappa and the MCC across
nights (the pooled values keep its epochs).

- **Inputs**: the recordings (`.edf`, BrainVision `.vhdr` or Curry `.cdt`), tool 2's channel configuration in `config_param/` (channel names, and the EOG,
  chin EMG and ECG declared as context channels in tool 2's Section 3) and, optionally, the expert hypnogram
  remapped by tool 3 (`{id}_hypno_remapped.txt`, or `{id}_Hypnogram_remapped.txt` in an older study). SleepFM also
  takes the respiratory channels, found by name (Thoracic, Abdominal, Flux/Airflow, SpO2, Pulse; a single belt named
  Respi, Resp or Belt is taken as abdominal).
- **Outputs**: per model, the automatic hypnogram `{id}_hypno_<algo>.txt` beside each recording, the hypnodensities in
  `derivatives/autoscoring_<algo>/` and the per-night and database reports in `reports_autoscoring_<algo>/`;
  Section 4 writes `derivatives/autoscoring_comparison/` and `reports_autoscoring_comparison/` (the full tree is in
  Details). The output suffixes are noted in
  `config_param/file_names.json`, so the other tools never take an automatic hypnogram for the expert one.
- **Sections**: 1. Study folder, 2. Models and channels, 3. Run, 4. Compare the models.

### The three models in a few words

| | What it is | Trained on | Needs | Works best on |
|---|---|---|---|---|
| **GSSC** | a deep neural network (Greifswald, 2023) | ~3,300 clinical nights of adults and children (NSRR) | 1 EEG and/or the EOG | clinical PSG; robust with few channels, even the EOG alone |
| **YASA** | decision trees on hand-made EEG measures (2021) | ~3,000 nights (NSRR), all ages and health conditions | 1 central EEG; EOG and chin EMG improve it | whole nights with a central EEG; fast and light |
| **SleepFM** | a "foundation model" of the whole PSG (Stanford, 2026) | over 585,000 hours of PSG, ~65,000 people | EEG/EOG, chin EMG, ECG and respiration | complete PSG montages (sleep staging is a side task of this model) |

All three were trained on standard clinical montages, where each EEG electrode is referenced to the mastoid of
the **other** side (C3-M2, C4-M1...). That is the reference given to them by default (Section 2), whatever
reference tool 2 declared for the other tools. The channels each model receives are checked (a noisy or dead
electrode, mastoid or EOG; an ECG or chin EMG far from its usual amplitude): a ⚠ in the log and the report, nothing
is changed.

<details style="border:1px solid #bbb; border-radius:4px; background:#f7f7f7; padding:8px 12px; margin:10px 0;"><summary style="cursor:pointer; font-weight:bold;">Details <span style="font-size:85%; font-weight:normal; color:#666;">(click to open)</span></summary>

**Environment**: GSSC, YASA and SleepFM run in the `somnolens` environment. SleepFM's model files (~29 MB,
official repository) are downloaded automatically the first time.

**Outputs** (`<algo>` = `gssc`, `yasa` or `sleepfm`):

```
{data_folder}/
  {id}_hypno_<algo>.txt                   ← automatic hypnogram beside the recording (tool-3 format: one label per line)
  derivatives/autoscoring_<algo>/<subtree>/
    {id}_hypnodensity.tsv                 ← 1 row/epoch: stage, p_W … p_R, confidence, expert stage
    {id}_scoring_params.json              ← channels, reference, versions (provenance)
    {id}_hypnodensity_5s.tsv              ← SleepFM only: its native 5-s probabilities
    {id}_sleepfm_features.npz             ← SleepFM only: its internal features, for recalibration B
  reports_autoscoring_<algo>/<subtree>/
    {id}_autoscoring_comparison.tsv       ← agreement metrics with the expert hypnogram (1 row)
    {id}_confusion.tsv                    ← confusion matrix, long format (expert, <algo>, n)
    {id}_autoscoring_report.html          ← hypnograms, hypnodensity, confusion matrix
  reports_autoscoring_<algo>/
    global_autoscoring_summary.tsv        ← every participant (rebuilt from the per-file tables on disk)
    global_autoscoring_pooled.tsv         ← agreement metrics of every participant pooled (1 row)
    global_confusion.tsv                  ← pooled confusion matrix
    global_autoscoring_report.html
  derivatives/autoscoring_comparison/     ← Section 4: the models compared (replaced by each Compare)
    recalibration_params.tsv              ← SleepFM's recalibrations: 1 row per method and held-out night
    <subtree>/{id}_comparison_epochs.tsv  ← 1 row per epoch and model: expert and model stages, p_W … p_R
    <subtree>/{id}_recal_b_layer.npz      ← recalibration B's refitted final layer for that night
  reports_autoscoring_comparison/
    <subtree>/{id}_comparison.tsv         ← agreement metrics and channels, 1 row per model
    <subtree>/{id}_confusion.tsv          ← confusion matrices, long format (model, expert, predicted, n)
    <subtree>/{id}_comparison_report.html ← hypnodensities of every model, agreement, confusion matrices
    global_comparison_summary.tsv         ← every night and model (rebuilt from the per-night tables)
    global_comparison_pooled.tsv          ← agreement metrics of every night pooled, 1 row per model
    global_confusion.tsv                  ← pooled confusion matrices
    global_comparison_report.html
```

**For experts: training, inputs, preprocessing, performance**

| | **YASA** | **GSSC** | **SleepFM-Clinical** |
|---|---|---|---|
| Reference | Vallat & Walker 2021, *eLife* | Hanna & Flöel 2023, *Front. Neuroinform.* | Thapa et al. 2026, *Nature Medicine* |
| Model | gradient-boosted trees (LightGBM) on hand-crafted features | 1-D ResNets (one EEG branch, one EOG branch) + bidirectional GRU (context of the night) | transformer pretrained by contrastive learning, then a sleep-staging head (transformer + bidirectional LSTM) |
| Training data | ~3,000 NSRR nights (varied ages, sexes, pathologies) | 3,314 NSRR nights: SHHS2 (936), CFS (730), NCHSDB (665), WSC (983); external test DREEM (80 subjects, consensus of 5 scorers) | pretraining: > 585,000 h of PSG, ~65,000 participants (Stanford Sleep Dataset released; SHHS left out) |
| EEG | **1 central, required**, on the mastoid (C4-M1 preferred, C3-M2) | C3, C4, F3, F4 on the contralateral mastoid (or Fpz); 1 or 2 EEG recommended | 0 to 10 "BAS" channels (EEG + EOG) |
| EOG | optional: left EOG (LOC/E1-M2) preferred | **HEOG = left EOG - right EOG** (left or right alone also works; VEOG not recommended) | part of BAS |
| EMG | optional: chin | not used | EMG modality: 1 to 4 channels |
| Other | age/sex optional (not used here) | none | **RESP** (1 to 7 channels: thorax, abdomen, flow, SpO2, pulse) and **EKG** (1 to 2) |
| Internal preprocessing | resampled to 100 Hz, band-pass 0.4-30 Hz, robust z-score over the night. **Do not filter before** | band-pass 0.3-30 Hz, z-score per channel and night, 85.33 Hz (2,560 samples per 30 s) | anti-alias low-pass, interpolation to 128 Hz, z-score per channel and night |
| Output | one stage per 30 s | one stage per 30 s | **one prediction per 5 s** (averaged over each 30-s epoch here) |
| Published performance | ~85 % agreement with the expert consensus; N1 the weakest | > 80 % agreement on every test set, including one EEG channel or the EOG alone | mean F1 0.70-0.78 for sleep staging (a secondary task: the authors say a specialised model would do better) |
| Licence | BSD-3 | **AGPL-3.0** | **CC BY-NC 4.0** (non-commercial use only) |

- **YASA** computes, per 30-s epoch and per channel: standard deviation, interquartile range, skewness,
  kurtosis, zero crossings, Hjorth parameters, absolute and relative band powers, power ratios, permutation
  entropy and fractal dimensions. It smooths them over 7.5 min (centred) and 2 min (past), then normalises them
  over the whole night: it needs **the whole night**, not pieces. Sampling rate above 80 Hz.
- **GSSC** uses a "permutation consensus": it scores the night with **every combination** of the signals it is
  given (each EEG with the EOG, each EEG alone, the EOG alone) and keeps, epoch by epoch, the most confident
  combination. With 4 EEG and 1 EOG that is 9 passes over the night. GSSC drops an incomplete last epoch, and also
  the last, complete, epoch of a recording that lasts a whole number of 30-s epochs (gssc 0.0.9 takes the time of
  the last sample as the duration; its own scoring function does the same; not in its documentation, reported as
  issue #9 of its repository, open since August 2025). That epoch gets no GSSC stage: a note in the log and the
  night's report says so, and it stays out of the comparison with the expert.
- **SleepFM** cuts each channel into 5-s tokens at 128 Hz; an attention layer pools the channels of a modality
  (hence any number of channels), a transformer works on 5-min chunks and the staging head (bidirectional LSTM)
  reads the whole night (up to 11.4 h). It needs **at least one channel in each of its 4 modalities** (BAS,
  RESP, EKG, EMG). The signals are band-passed first (EEG/EOG 0.3-35 Hz, chin EMG 10-100 Hz): without it 50-Hz
  mains passes its filters and the model gives almost no REM.
- **SleepFM recalibration A** learns a "temperature" and one bias per stage (Wake is the reference) on the
  other nights, by leave-one-night-out: at least 2 nights. **Recalibration B** refits SleepFM's final layer on
  its frozen internal features, its penalty chosen by an inner cross-validation: at least 3 nights (6 or more
  advised). Both learn from **your expert hypnograms** (a supervised adaptation); each night is predicted with
  parameters learnt without it, so the comparison stays fair.

</details>

In [ ]:
try:
    import os
    import json
    import re
    import time
    import datetime
    import io
    import shutil
    import warnings
    from pathlib import Path

    import numpy as np
    import pandas as pd
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    import mne
    from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score, confusion_matrix

    import ipywidgets as widgets
    from IPython.display import display, HTML
    from ipyfilechooser import FileChooser
except ImportError as e:
    print("❌ Import error, a package is missing:", e)

warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')

# Shared libraries (tools/participant_selection_lib.py, tools/autoscoring_lib.py), probed from the repo root (the
# canonical launch folder) or from tools/, so the import works whatever folder Voila is launched from.
import sys
for _p in [os.getcwd(), os.path.join(os.getcwd(), 'tools'),
           os.path.join(os.path.dirname(os.getcwd()), 'tools'), os.path.join(os.getcwd(), '..', 'tools')]:
    if os.path.isfile(os.path.join(_p, 'participant_selection_lib.py')):
        if _p not in sys.path:
            sys.path.insert(0, _p)
        break
import participant_selection_lib as PSL
import ui_lib as UI   # the look shared by the Voila pages
import study_files_lib as SF   # the names registry and the choice of the companion files

In [ ]:
# The computation and the figures of this tool live in shared libraries: tools/autoscoring_lib.py (step 1,
# shared with tool 0), tools/sleepfm_lib.py (SleepFM) and tools/autoscoring_compare_lib.py (step 2, Section 4).
from autoscoring_lib import (
    GSSC_OK, GSSC_MSG, GSSC_VERSION, TORCH_VERSION, CONSENSUS_RULE, EPOCH_SEC, STAGES,
    HYPNODENSITY_ROWS_PER_FIG, agreement_metrics, clock_hours, compare_scoring, eeg_options,
    load_custom_stages, load_expert, load_json_lenient, metrics_html, non_aasm_profile, params_html,
    plot_confusion, plot_hypnodensity, plot_hypnodensity_stack, plot_hypnograms, plot_non_aasm_profile,
    pool_profiles, recording_start, save_report_html,
    ALGORITHMS, EOG_MODES, REFERENCE_OPTIONS, REFERENCE_CONTRALATERAL, YASA_OK, YASA_MSG, YASA_VERSION, ANTROPY_VERSION,
    non_aasm_help, score_night_gssc, score_night_sleepfm, score_night_yasa, YASA_EMG_DECLARED,
    scoring_settings_mismatch, stored_scoring_settings, model_context_gain, context_gain_text,
    eeg_counts, eog_counts, menu_options, menu_default, missing_channel_lines, sleepfm_channels_text,
    GSSC_EOG_ROLES, YASA_EOG_ROLES, GSSC_EOG_PREFERRED, YASA_EOG_PREFERRED)
import recording_lib as RL   # one way to read a recording (EDF, BrainVision, Curry)
import sleepfm_lib as SFM
import autoscoring_compare_lib as ACL
if GSSC_OK:
    import torch
    from gssc.infer import EEGInfer
    UI.log(f"✅ GSSC {GSSC_VERSION} (torch {TORCH_VERSION}) ready"
          f"{', GPU available' if torch.cuda.is_available() else ', running on CPU'}.")
else:
    display(HTML(GSSC_MSG))
if YASA_OK:
    UI.log(f"✅ YASA {YASA_VERSION} ready.")
else:
    display(HTML(UI.msg('error', YASA_MSG, symbol=False)))
if SFM.SLEEPFM_DEPS_OK:
    UI.log("✅ SleepFM can run (its model is checked in Section 2).")
else:
    display(HTML(UI.msg('warn', SFM.SLEEPFM_DEPS_MSG)))

<hr style="height:4px; background-color:black; border:none;">

## 1. Study folder

Select the folder holding the recordings (`.edf`, BrainVision `.vhdr` or Curry `.cdt`, searched recursively), then click **Scan**. The tool finds tool 2's channel
configuration in it and lists it under **Inputs found**; *Choose another…* on its line picks another file by hand. The
**expert hypnogram suffix** is detected automatically (leave it empty when there is no expert hypnogram: the nights are
then scored without comparison). The **Advanced** box holds the **output suffixes**: they name the automatic hypnogram
each model writes beside each recording.

In [ ]:
fc_data = FileChooser()
fc_data.title = UI.picker_title('Study folder')
fc_data.show_only_dirs = True

hypno_suffix = widgets.Text(value=SF.HYPNO_REMAPPED_DEFAULT, description='Expert hypno suffix:',
                            style=UI.LABEL_STYLE, layout=widgets.Layout(width=UI.FIELD_WIDTH))
hypno_suffix_info = widgets.HTML(value='')
# A night where the expert scored one stage only (a nap where the participant stays awake) has a kappa and an MCC
# of 0 whatever the model: ticked, it is left out of what summarises several nights (pooled values keep it).
cb_single_stage_out = widgets.Checkbox(
    value=True, indent=False, layout=widgets.Layout(width=UI.WIDE_WIDTH),
    description='Leave single-stage nights out of the kappa and MCC across nights')
single_stage_help = widgets.HTML(UI.help_html(
    'A night where the expert scored a single stage (a nap where the participant stays awake) has a kappa and an '
    'MCC of 0 whatever the model, by construction. Ticked, such nights are left out of the kappa and the MCC '
    'summarised across nights (database reports of Section 3, Section 4); the values with every night pooled keep '
    'their epochs, and each night keeps its own values in its tables.'))
# a menu of the detected suffixes in front of the expert suffix field (the field stays the value the run reads)
hypno_menu, fill_hypno_menu = UI.suffix_menu(hypno_suffix, hypno_suffix_info)
# One output suffix per model: the automatic hypnogram each one writes beside the EDF.
out_suffix = widgets.Text(value=SF.hypno_auto_default('gssc'), description='GSSC output suffix:',
                          style=UI.LABEL_STYLE, layout=widgets.Layout(width=UI.FIELD_WIDTH))
out_suffix_yasa = widgets.Text(value=SF.hypno_auto_default('yasa'), description='YASA output suffix:',
                               style=UI.LABEL_STYLE, layout=widgets.Layout(width=UI.FIELD_WIDTH))
out_suffix_sleepfm = widgets.Text(value=SF.hypno_auto_default('sleepfm'), description='SleepFM output suffix:',
                                  style=UI.LABEL_STYLE, layout=widgets.Layout(width=UI.FIELD_WIDTH))
OUT_SUFFIX = {'GSSC': out_suffix, 'YASA': out_suffix_yasa, 'SleepFM': out_suffix_sleepfm}
ALGO_KEY = {'GSSC': 'gssc', 'YASA': 'yasa', 'SleepFM': 'sleepfm'}   # folder and column names
_proposed_out = {}         # the output suffixes proposed for this study: the default of the Advanced box's fields
# The input file, found in the study folder (the 'Inputs found' list): its picker shows under its line, with
# "Choose another…", for a file chosen by hand. The scan reads its selection.
fc_config = FileChooser(filter_pattern='*.json')
fc_config.title = UI.picker_title('Channel configuration', '(the JSON file tool 2 saved in <code>config_param/</code>)')
config_row, locate_config = UI.input_row(fc_config, 'Channel configuration', 'required', 'run tool 2 first',
                                         check=SF.check_channel_config)
btn_scan = UI.button('Scan', 'read')
scan_info = widgets.HTML(value='')
participant_selector = PSL.ParticipantSelector()

STATE = {'cfg': {}, 'edf_by_id': {}, 'not_in_cfg': set(), 'custom': []}


def _detect_hypno_suffix(chooser):
    """Auto-fill the output suffixes (a study keeps the names it already has) and the expert hypnogram suffix
    (study_files_lib.pick_hypno_suffix: the name tool 3 recorded, else the longest among the .txt suffixes found for
    >= 50 % of the most frequent one; the exports, the toolkit's hypnograms and the outputs below are set aside)."""
    try:
        fill_hypno_menu({}, [])   # emptied until the detection below
        if not chooser.selected:
            show_inputs(False)
            locate_config(None)
            hypno_suffix_info.value = ''
            return
        folder = Path(chooser.selected)
        show_inputs(True)
        names, names_warn = SF.load_names(folder)
        locate_config(folder, *SF.find_input(folder, names, 'channel_config'))
        edf_files = SF.list_recordings(folder)
        if not edf_files:
            hypno_suffix_info.value = (f'<small style="color:{UI.MUTED_COLOR};">No recording found (recursive scan: '
                                       '.edf, .vhdr, .cdt).</small>')
            return
        all_txt = [f for f in folder.rglob('*') if f.suffix.lower() == '.txt']
        owners = SF.companion_suffix_owners(edf_files, all_txt)
        counts = {s: len(r) for s, r in owners.items()}
        # a study keeps its names: the suffix each model wrote before, else the short default
        for algo, widget in OUT_SUFFIX.items():
            key = ALGO_KEY[algo]
            _proposed_out[algo] = SF.output_suffix(names, f'hypnogram_{key}', counts, SF.hypno_auto_default(key),
                                                   SF.hypno_auto_former(key))
            widget.value = _proposed_out[algo]
        _update_advanced_mark()
        best, best_n, reason = SF.pick_hypno_suffix(counts, 'expert', names,
                                                    own_outputs=[w.value for w in OUT_SUFFIX.values()])
        if best is None:
            hypno_suffix.value = ''
            fill_hypno_menu(owners, [f.stem for f in edf_files])      # a toolkit hypnogram can still be chosen
            hypno_suffix_info.value = (f'<small style="color:{UI.WARN_COLOR};">No expert hypnogram detected: the '
                                       'nights will be scored without comparison.</small>'
                                       + (f'<br><small>{UI.msg("warn", names_warn)}</small>' if names_warn else ''))
            return
        hypno_suffix.value = best
        fill_hypno_menu(owners, [f.stem for f in edf_files],
                        ''.join(f'<br><small>{UI.msg(k, t)}</small>'
                                for k, t in SF.choice_notes(reason, 'expert', names_warn)))
    except Exception as e:
        hypno_suffix_info.value = f'<small style="color:{UI.ERROR_COLOR};">Error detecting hypnograms: {e}</small>'


def _update_advanced_mark(change=None):
    """'⚠️ default changed' in the title of the Advanced box when an output suffix differs from the one proposed."""
    mark_advanced([f'{algo} output suffix' for algo, widget in OUT_SUFFIX.items()
                   if widget.value != _proposed_out.get(algo, widget.value)])


fc_data.register_callback(_detect_hypno_suffix)
for _widget in OUT_SUFFIX.values():
    _widget.observe(_update_advanced_mark, names='value')


def output_dirs(data_root, edf_path, algo='GSSC'):
    rel = edf_path.parent.relative_to(data_root)
    key = ALGO_KEY[algo]
    return (data_root / 'derivatives' / f'autoscoring_{key}' / rel,
            data_root / f'reports_autoscoring_{key}' / rel)


def output_paths(data_root, edf_path, fid, algo='GSSC'):
    deriv, reports = output_dirs(data_root, edf_path, algo)
    paths = {'hypno': edf_path.parent / f'{fid}{OUT_SUFFIX[algo].value}',
             'density': deriv / f'{fid}_hypnodensity.tsv',
             'params': deriv / f'{fid}_scoring_params.json',
             'comparison': reports / f'{fid}_autoscoring_comparison.tsv',
             'confusion': reports / f'{fid}_confusion.tsv',
             'report': reports / f'{fid}_autoscoring_report.html'}
    if algo == 'SleepFM':
        paths['density_5s'] = deriv / f'{fid}_hypnodensity_5s.tsv'      # native 5-s probabilities
        paths['features'] = deriv / f'{fid}_sleepfm_features.npz'       # for recalibration B (Section 4)
    return paths


def algo_outputs(data_root, edf_path, fid, algo):
    """(complete, partial) for one model's outputs: complete = automatic hypnogram + hypnodensity (+ SleepFM's
    5-s table and features) + report all on disk (the data are written before the report)."""
    p = output_paths(data_root, edf_path, fid, algo)
    keys = ['hypno', 'density', 'report'] + (['density_5s', 'features'] if algo == 'SleepFM' else [])
    have = [p[k].exists() for k in keys]
    return all(have), any(have) and not all(have)


def refresh_done():
    """'Already processed' = every ticked model's outputs on disk (recomputed when the models change).
    Returns the participants with incomplete outputs of a ticked model (they are rescored)."""
    if not fc_data.selected or not STATE['edf_by_id']:
        return []
    data_root = Path(fc_data.selected)
    algos = ticked_algorithms()
    done, mismatch = [], []
    for fid, edf in STATE['edf_by_id'].items():
        if fid not in STATE['cfg']:
            continue
        states = [algo_outputs(data_root, edf, fid, a) for a in algos]
        if algos and all(complete for complete, _ in states):
            done.append(fid)
        elif any(partial for _, partial in states):
            mismatch.append(fid)
    participant_selector.set_participants(list(STATE['edf_by_id']), done=done)
    STATE['n_done'] = len(done)
    return mismatch


def on_scan(btn):
    btn.disabled = True
    try:
        if not fc_data.selected:
            scan_info.value = f'<span style="color:{UI.ERROR_COLOR};">Select the study folder first.</span>'
            return
        data_root = Path(fc_data.selected)
        if not fc_config.selected or not Path(fc_config.selected).exists():
            scan_info.value = UI.msg('error', 'No channel configuration: run tool 2 (2_select&amp;remap_channels) '
                                              'first, or choose one with "Choose another…".')
            return
        cfg_path = Path(fc_config.selected)
        try:
            cfg = load_json_lenient(cfg_path)
        except Exception as e:
            scan_info.value = f'<span style="color:{UI.ERROR_COLOR};">Cannot read {cfg_path.name}: {e}</span>'
            return
        edf_files = SF.list_recordings(data_root)
        cfg_keys = {os.path.normcase(k): k for k in cfg}
        STATE['cfg'] = {}
        STATE['edf_by_id'] = {}
        STATE['not_in_cfg'] = set()
        n_eog = 0
        options = []
        for edf in edf_files:
            fid = edf.stem
            STATE['edf_by_id'][fid] = edf
            key = cfg_keys.get(os.path.normcase(fid))
            if key is None:
                STATE['not_in_cfg'].add(fid)
                continue
            STATE['cfg'][fid] = cfg[key]
            ctx = cfg[key].get('context_channels') or {}
            if ctx.get('eog_left') or ctx.get('eog_right'):
                n_eog += 1
            for ch in eeg_options(cfg[key]):
                if ch not in options:
                    options.append(ch)
        # Every channel name of the recordings (header only): the choices of YASA's chin EMG (Section 2)
        channel_names = set()
        STATE['file_channels'] = {}       # the channels of each file: the menus say which recordings have them
        for edf in edf_files:
            try:
                header = RL.read_header(edf)
                # a BrainVision online reference is listed in its header but is not in the file (in_file False)
                in_file = header.get('in_file', [True] * len(header['channel']))
                in_the_file = [ch for ch, kept in zip(header['channel'], in_file) if kept]
                STATE['file_channels'][edf.stem] = in_the_file
                channel_names.update(in_the_file)
            except Exception:
                pass
        set_emg_options(sorted(channel_names))
        mismatch = refresh_done()
        STATE['custom'] = load_custom_stages(data_root)
        set_eeg_options(options)
        set_eog_options()
        refresh_channel_lines()
        n_cfg = len(STATE['cfg'])
        msg = (f'<span style="color:{UI.OK_COLOR};">{len(edf_files)} recording(s): {n_cfg} configured in tool 2, '
               f'{n_eog} with an EOG declared (context channels), {STATE["n_done"]} already scored'
               f' ({", ".join(ticked_algorithms()) or "no model ticked"}).</span>')
        if STATE['custom']:
            msg += (f'<br><span style="color:{UI.MUTED_COLOR};">Custom stage(s) from tool 3: {", ".join(STATE["custom"])} '
                    '(shown in a dedicated report section, outside the agreement metrics).</span>')
        if STATE['not_in_cfg']:
            msg += (f'<br><span style="color:{UI.WARN_COLOR};">⚠️ {len(STATE["not_in_cfg"])} recording(s) not in the '
                    f'tool-2 JSON, will not be scored: {", ".join(sorted(STATE["not_in_cfg"]))}.</span>')
        if n_cfg and n_eog < n_cfg:
            msg += (f'<br><span style="color:{UI.WARN_COLOR};">⚠️ {n_cfg - n_eog} configured participant(s) without '
                    'EOG: scored on EEG only (declare the EOG in tool 2, Section 3).</span>')
        if mismatch:
            msg += (f'<br><span style="color:{UI.WARN_COLOR};">⚠️ {len(mismatch)} participant(s) with incomplete '
                    f'outputs, will be rescored: {", ".join(mismatch)}.</span>')
        conflicts = SF.recording_conflicts_text(data_root)
        if conflicts:
            msg += f'<br>{UI.msg("warn", conflicts)}'
        scan_info.value = msg
    except Exception as e:
        scan_info.value = f'<span style="color:{UI.ERROR_COLOR};">Scan error: {e}</span>'
    finally:
        btn.disabled = False


btn_scan.on_click(on_scan)

advanced, mark_advanced = UI.advanced_box([out_suffix, out_suffix_yasa, out_suffix_sleepfm])
inputs_list, show_inputs = UI.inputs_list([config_row, hypno_menu, hypno_suffix, hypno_suffix_info,
                                            cb_single_stage_out, single_stage_help, advanced])
display(fc_data, inputs_list, btn_scan, scan_info, widgets.HTML('<b>Participants to score:</b>'),
        participant_selector.widget)

<hr style="height:4px; background-color:black; border:none;">

## 2. Models and channels

Tick the **models** to run. Each scores every selected night on its own and writes its own outputs.

**EEG reference**: by default each EEG electrode is referenced to the mastoid of the other side (C3-M2, C4-M1,
F3-M2...), the montage the three models were trained on. It needs both mastoids (M1/M2, A1/A2, or TP9/TP10 on a
10-10 cap) in tool 2's configuration; a participant without them is scored with tool 2's reference (said in the log). *As declared in
tool 2* uses the reference of the other tools instead.

**EOG recorded against the online reference**: an EOG electrode on an EEG input of the amplifier (BrainVision: µV,
no gradient in the header, e.g. IO against Cz) also carries the EEG of the reference electrode, inverted, which the
models never saw. It gets the reference of the EEG (IO-M1 for a right EOG with the contralateral mastoid), written in
the parameters and the report. An EOG on an auxiliary input, the L − R derivation of two EOG, and any EDF are used as
recorded.

**Channel checks** (60 epochs spread over the night): an electrode, a mastoid or an EOG noisy or dead (compared with
the median EEG channel of the same epoch), an ECG or a chin EMG far from its usual amplitude (a gain the file does
not declare) or dominated by the mains. A ⚠ in the log and the report: the night is scored anyway, nothing is changed.

**GSSC**. *Several electrodes (consensus)* (default, as Score_EDF did): GSSC scores with every ticked EEG and
the EOG, and keeps the most confident combination epoch by epoch, so a bad electrode on an epoch is outvoted
by a better one. Ticked by default: the electrodes GSSC was mostly trained on (C3, C4, F3, F4) that the
recordings have. A participant missing some of them is scored on the ones present. *One electrode (fast)*:
GSSC scores with that EEG and the EOG; a participant who does not have the electrode is **not** scored
(listed in the log): no other electrode is silently taken instead. EOG: the left minus right derivation
(default, what GSSC was mostly trained on), both sides separately, one side only, or none.

**YASA**: one EEG electrode (YASA prefers C4, else C3), one EOG (the left one is YASA's recommendation) and one
chin EMG: the one declared in tool 2 by default, or a channel picked in the list (the same name is then used
for every recording; a recording without it is scored without EMG, said in the log).

**SleepFM**: the AASM EEG derivations present (F3, F4, C3, C4, O1, O2), both EOG, the chin EMG and the ECG
declared in tool 2, and the respiratory channels found by name (a single belt named Respi, Resp or Belt counts as
abdominal). Its model is downloaded the first time.

The models filter and resample the signals themselves: no preprocessing is needed. On CPU, count about 1-3 min
per night for GSSC (more in consensus mode), under 1 min for YASA, several minutes for SleepFM.

In [ ]:
# ---- Models ----
cb_gssc = widgets.Checkbox(value=GSSC_OK, description='GSSC', disabled=not GSSC_OK, indent=False,
                           layout=widgets.Layout(width='120px'))
cb_yasa = widgets.Checkbox(value=False, description='YASA', disabled=not YASA_OK, indent=False,
                           layout=widgets.Layout(width='120px'))
cb_sleepfm = widgets.Checkbox(value=False, description='SleepFM', disabled=not SFM.SLEEPFM_DEPS_OK, indent=False,
                              layout=widgets.Layout(width='120px'))
ALGO_CHECKS = {'GSSC': cb_gssc, 'YASA': cb_yasa, 'SleepFM': cb_sleepfm}


def ticked_algorithms():
    return [a for a, cb in ALGO_CHECKS.items() if cb.value]


dd_ref = widgets.Dropdown(options=REFERENCE_OPTIONS, value=REFERENCE_CONTRALATERAL, description='EEG reference:',
                          style=UI.LABEL_STYLE, layout=widgets.Layout(width=UI.WIDE_WIDTH))

# ---- GSSC ----
# Consensus by default (Score_EDF's choice): over the electrodes GSSC was mostly trained on, when present.
GSSC_TRAINED_EEG = ('C3', 'C4', 'F3', 'F4')
tb_mode = widgets.ToggleButtons(options=['One electrode (fast)', 'Several electrodes (consensus)'],
                                value='Several electrodes (consensus)', style={'button_width': '230px'})
dd_eeg = widgets.Dropdown(options=[], description='EEG electrode:', style={'description_width': '150px'},
                          layout=widgets.Layout(width='360px',
                                                display='' if tb_mode.value.startswith('One') else 'none'))
# Consensus mode: one checkbox per electrode (filled by the scan), plus tick-all / untick-all buttons.
eeg_checks = {}                                   # electrode name -> Checkbox, in the scan's order
box_eeg_checks = widgets.HBox([], layout=widgets.Layout(flex_flow='row wrap', margin='0 0 0 12px'))
btn_eeg_all = UI.button('Tick all', 'helper')
btn_eeg_none = UI.button('Untick all', 'helper')
consensus_hint = widgets.HTML('')
box_consensus = widgets.VBox(
    [widgets.HTML('<b>EEG electrodes:</b>'), box_eeg_checks, widgets.HBox([btn_eeg_all, btn_eeg_none]),
     consensus_hint],
    layout=widgets.Layout(display='none' if tb_mode.value.startswith('One') else ''))


def selected_eeg():
    """Electrodes ticked for the consensus mode, in the scan's order."""
    return [ch for ch, cb in eeg_checks.items() if cb.value]


def _tick_all(value):
    for cb in eeg_checks.values():
        cb.value = value


btn_eeg_all.on_click(lambda _: _tick_all(True))
btn_eeg_none.on_click(lambda _: _tick_all(False))
dd_eog = widgets.Dropdown(options=EOG_MODES, value='L − R bipolar',
                          description='EOG:', style={'description_width': '150px'},
                          layout=widgets.Layout(width='360px'))
eeg_hint = widgets.HTML(f'<small style="color:{UI.MUTED_COLOR};">Run the scan (Section 1) to list the electrodes.</small>')


def _update_consensus_hint(change=None):
    """Why these electrodes are ticked, and what each extra one costs: GSSC scores the whole night once per
    combination (each EEG or none x each EOG or none, minus 'nothing')."""
    n = len(selected_eeg())
    eog_choices = {'None': 1, 'L and R separately': 3}.get(dd_eog.value, 2)
    passes = (n + 1) * eog_choices - 1
    one = 2 * eog_choices - 1                       # one electrode + the EOG
    color = UI.WARN_COLOR if n > 4 else UI.MUTED_COLOR
    consensus_hint.value = (
        f'<small style="color:{UI.MUTED_COLOR};">Ticked by default: the electrodes GSSC was mostly trained on (C3, C4, F3, '
        'F4) found in the recordings. Every extra electrode adds passes over the whole night: tick only a few '
        '(2 to 4) to keep the computing time reasonable.</small><br>'
        f'<small style="color:{color};">{n} electrode(s) ticked: {passes} combination(s), i.e. {passes} passes '
        f'over each night (about {passes / one:.0f}x the one-electrode mode).</small>')


dd_eog.observe(_update_consensus_hint, names='value')
box_gssc = widgets.VBox([widgets.HTML('<b>GSSC</b>'), tb_mode, dd_eeg, box_consensus, eeg_hint, dd_eog],
                        layout=widgets.Layout(display='' if cb_gssc.value else 'none', margin='6px 0'))

# ---- YASA ----
dd_yasa_eeg = widgets.Dropdown(options=[], description='EEG electrode:', style={'description_width': '150px'},
                               layout=widgets.Layout(width='360px'))
dd_yasa_eog = widgets.Dropdown(options=['Left', 'Right', 'None'], value='Left', description='EOG:',
                               style={'description_width': '150px'}, layout=widgets.Layout(width='360px'))
dd_yasa_emg = widgets.Dropdown(options=[YASA_EMG_DECLARED, 'None'], value=YASA_EMG_DECLARED,
                               description='Chin EMG:', style={'description_width': '150px'},
                               layout=widgets.Layout(width='460px'))


def set_emg_options(names):
    """YASA's chin EMG choices after a scan: tool 2's declared EMG (default), none, then the channels of the
    recordings, the EMG-like names first; '(n/N)' after a choice that not every recording configured in tool 2 has.
    A rescan keeps the user's choice when still offered."""
    previous = dd_yasa_emg.value
    configs, files = STATE.get('cfg', {}), STATE.get('file_channels', {})
    emg_like = [n for n in names if re.search(r'emg|chin|menton|submental', n, re.IGNORECASE)]
    counts = {YASA_EMG_DECLARED: sum(bool((c.get('context_channels') or {}).get('emg')) for c in configs.values()),
              'None': len(configs)}
    for name in emg_like + [n for n in names if n not in emg_like]:
        counts[name] = sum(name in set(files.get(f, ())) for f in configs)
    dd_yasa_emg.options = menu_options(counts, len(configs), always=(YASA_EMG_DECLARED, 'None'))
    values = [v for _, v in dd_yasa_emg.options]
    dd_yasa_emg.value = previous if previous in values else YASA_EMG_DECLARED


box_yasa = widgets.VBox(
    [widgets.HTML('<b>YASA</b>'), dd_yasa_eeg, dd_yasa_eog, dd_yasa_emg],
    layout=widgets.Layout(display='' if cb_yasa.value else 'none', margin='6px 0'))

# ---- SleepFM: its model, found on disk or downloaded from the official repository ----
sleepfm_status = widgets.HTML('')
btn_sleepfm_download = UI.button('Download SleepFM now', 'run', icon='download')
sleepfm_progress, sleepfm_progress_row = UI.progress_bar('Download (MB):')
sleepfm_progress.max = max(1, SFM.SLEEPFM_TOTAL_BYTES // 1_000_000)
sleepfm_progress_row.layout.display = 'none'
sleepfm_progress_label = widgets.HTML('')
box_sleepfm = widgets.VBox(
    [widgets.HTML(f'<b>SleepFM</b> <small style="color:{UI.MUTED_COLOR};">(EEG derivations, both EOG, chin EMG and ECG '
                  'from tool 2; respiration found by name)</small>'),
     sleepfm_status, btn_sleepfm_download, sleepfm_progress_row, sleepfm_progress_label],
    layout=widgets.Layout(display='' if cb_sleepfm.value else 'none', margin='6px 0'))


def refresh_sleepfm_status():
    root, how = SFM.find_sleepfm()
    if root is not None:
        sleepfm_status.value = f'<small style="color:{UI.OK_COLOR};">✅ SleepFM model found: {how}.</small>'
        btn_sleepfm_download.layout.display = 'none'
    else:
        sleepfm_status.value = (f'<small style="color:{UI.WARN_COLOR};">SleepFM model {how}: it will be downloaded '
                                f'(~{SFM.SLEEPFM_TOTAL_BYTES / 1e6:.0f} MB, official repository, version '
                                f'{SFM.SLEEPFM_COMMIT[:8]}) when you click Run, or now with the button. '
                                f'Destination: {SFM.cache_root()}</small>')
        btn_sleepfm_download.layout.display = ''
    return root


def ensure_sleepfm(say):
    """The SleepFM folder, downloaded first when needed (progress bar + messages). None if it failed."""
    root, how = SFM.find_sleepfm()
    if root is not None:
        return root
    say(f'Downloading the SleepFM model (~{SFM.SLEEPFM_TOTAL_BYTES / 1e6:.0f} MB) from the official '
        f'repository into {SFM.cache_root()} ...')
    sleepfm_progress_row.layout.display = ''
    sleepfm_progress.value = 0

    def show(done, total, name):
        sleepfm_progress.value = done // 1_000_000
        sleepfm_progress_label.value = (f'<small>{done / 1e6:.1f} / {total / 1e6:.1f} MB — '
                                        f'{name.split("/")[-1]}</small>')
    try:
        root = SFM.download_sleepfm(progress=show)
        say(f'✅ SleepFM downloaded and checked (every file matches its fingerprint): {root}')
        return root
    except Exception as e:
        say(f'❌ SleepFM download failed: {e}')
        return None
    finally:
        refresh_sleepfm_status()


def _on_download(btn):
    btn.disabled = True
    try:
        ensure_sleepfm(lambda m: setattr(sleepfm_progress_label, 'value', f'<small>{m}</small>'))
    finally:
        btn.disabled = False


btn_sleepfm_download.on_click(_on_download)
refresh_sleepfm_status()


def set_eeg_options(options):
    """Fill the EEG pickers after a scan. GSSC one-electrode mode: C3, else C4, else the first electrode;
    consensus: C3, C4, F3, F4 ticked (those present; else the first electrode).
    YASA: C4 (its preference), else C3, else the first. A rescan keeps the user's choices still valid."""
    previous_one = dd_eeg.value
    previous_ticks = {ch: cb.value for ch, cb in eeg_checks.items()}
    previous_yasa = dd_yasa_eeg.value
    # '(n/N)' after an electrode that not every recording configured in tool 2 has
    counts = eeg_counts(STATE.get('cfg', {}))
    labelled = menu_options({ch: counts.get(ch, 0) for ch in options}, len(STATE.get('cfg', {})), always=options)
    dd_eeg.options = labelled
    dd_yasa_eeg.options = labelled
    eeg_checks.clear()
    trained = [ch for ch in options if ch.upper() in GSSC_TRAINED_EEG]
    for label, ch in labelled:
        default = ch in trained or (not trained and ch == options[0])
        eeg_checks[ch] = widgets.Checkbox(value=previous_ticks.get(ch, default), description=label, indent=False,
                                          layout=widgets.Layout(width='auto', margin='2px 16px 2px 0'))
    box_eeg_checks.children = list(eeg_checks.values())
    for cb in eeg_checks.values():
        cb.observe(_update_consensus_hint, names='value')
        cb.observe(refresh_channel_lines, names='value')
    _update_consensus_hint()
    if options:
        dd_eeg.value = (previous_one if previous_one in options
                        else next((c for c in ('C3', 'C4') if c in options), options[0]))
        dd_yasa_eeg.value = (previous_yasa if previous_yasa in options
                             else next((c for c in ('C4', 'C3') if c in options), options[0]))
        eeg_hint.value = f'<small style="color:{UI.MUTED_COLOR};">{len(options)} EEG electrode(s) available.</small>'
    else:
        eeg_hint.value = f'<small style="color:{UI.ERROR_COLOR};">No EEG electrode found in the tool-2 JSON.</small>'


def set_eog_options():
    """The EOG menus after a scan: only the choices tool 2's context channels allow, '(n/N)' after one that not every
    recording allows; the previous choice kept when every recording allows it, else the first that every recording
    allows (GSSC: L - R bipolar, left, right; YASA: left, right), else the one most recordings allow."""
    configs = STATE.get('cfg', {})
    for dd, roles, preferred in ((dd_eog, GSSC_EOG_ROLES, GSSC_EOG_PREFERRED),
                                 (dd_yasa_eog, YASA_EOG_ROLES, YASA_EOG_PREFERRED)):
        counts = eog_counts(configs, roles)
        previous = dd.value
        dd.options = menu_options(counts, len(configs))
        dd.value = menu_default(counts, len(configs), previous, preferred)


channels_info = widgets.HTML('')


def refresh_channel_lines(change=None):
    """Under the models: the recordings that lack a channel chosen above, and what they get (autoscoring_lib)."""
    try:
        configs = STATE.get('cfg', {})
        one_mode = tb_mode.value.startswith('One')
        lines = missing_channel_lines(configs, ticked_algorithms(),
                                      gssc_eeg=[dd_eeg.value] if one_mode else selected_eeg(), gssc_one=one_mode,
                                      gssc_eog=dd_eog.value, yasa_eeg=dd_yasa_eeg.value, yasa_eog=dd_yasa_eog.value,
                                      yasa_emg=dd_yasa_emg.value, file_channels=STATE.get('file_channels', {}))
        channels_info.value = UI.text_block('<br>'.join(UI.msg('warn', f'{model}: {text}') for model, text in lines))
    except Exception as e:
        channels_info.value = UI.msg('warn', f'Could not check the channels of the recordings: {e}')


for _w in (dd_eeg, dd_eog, dd_yasa_eeg, dd_yasa_eog, dd_yasa_emg, tb_mode):
    _w.observe(refresh_channel_lines, names='value')


def _on_mode(change):
    one = change['new'].startswith('One')
    dd_eeg.layout.display = '' if one else 'none'
    box_consensus.layout.display = 'none' if one else ''


tb_mode.observe(_on_mode, names='value')


def _on_algorithms(change=None):
    box_gssc.layout.display = '' if cb_gssc.value else 'none'
    box_yasa.layout.display = '' if cb_yasa.value else 'none'
    box_sleepfm.layout.display = '' if cb_sleepfm.value else 'none'
    refresh_channel_lines()
    try:
        refresh_done()              # 'already processed' depends on the models ticked
    except Exception as e:
        scan_info.value += f'<br><span style="color:{UI.ERROR_COLOR};">{e}</span>'


for _cb in ALGO_CHECKS.values():
    _cb.observe(_on_algorithms, names='value')

display(widgets.HTML('<b>Models:</b>'), widgets.HBox([cb_gssc, cb_yasa, cb_sleepfm]), dd_ref,
        box_gssc, box_yasa, box_sleepfm, channels_info)

<hr style="height:4px; background-color:black; border:none;">

## 3. Run

Scores the selected participants with each ticked model, writes their outputs, then rebuilds each model's
database summary from every per-participant table on disk (so a participant scored in an earlier run is still
included). With *Skip already processed* ticked, a model whose outputs are already on disk for a participant
is not run again.

In [ ]:
btn_run = UI.button('Run automatic scoring', 'run')
progress, progress_row = UI.progress_bar('Participants:')
# the models of one participant, a segment each (their time on a night: GSSC and SleepFM about 40 s, YASA 8 s)
progress_model, progress_model_row, set_model_phases, lbl_model = UI.phase_bar('Models:')
MODEL_COST = {'GSSC': 40, 'YASA': 8, 'SleepFM': 40}
out_run = widgets.Output()
# Expert switch (no widget): True predicts SleepFM's last minutes with the whole night as context, as
# Score_EDF did (slower); False uses 30 min of context (sleepfm_lib.TAIL_CONTEXT_CHUNKS).
SLEEPFM_FULL_TAIL = False
# What the report says under the channel warnings: the ways out that already exist (nothing is changed automatically).
CHECKS_ADVICE = ('The night was scored with these channels: nothing is changed automatically. To score without a bad '
                 'channel: another electrode in Section 2 (the electrodes of the other side when a mastoid is bad), '
                 'EEG reference = As declared in tool 2, or EOG = None; then rescore this participant alone '
                 '(participant selector, Skip unticked).')


def log(msg):
    with out_run:
        UI.log(msg)        # a line with a symbol in its colour


def expert_path(edf_path, fid):
    """The expert hypnogram of a participant (Section 1 suffix), or None."""
    if not hypno_suffix.value.strip():
        return None
    hp = edf_path.parent / f'{fid}{hypno_suffix.value}'
    if not hp.exists():
        hp = next((p for p in edf_path.parent.iterdir()
                   if os.path.normcase(p.name) == os.path.normcase(hp.name)), hp)
    return hp if hp.exists() else None


def run_settings(algo, edf_path, fid, cfg_entry):
    """This run's settings for one night, in the words of the library's stored_scoring_settings (same keys
    and values), to compare with those of a night already scored."""
    settings = {'reference': dd_ref.value,
                'expert hypnogram': hypno_suffix.value if expert_path(edf_path, fid) is not None else 'none',
                'context channels divided by': context_gain_text(model_context_gain(algo, cfg_entry))}
    if algo == 'GSSC':
        one_mode = tb_mode.value.startswith('One')
        wanted = [dd_eeg.value] if one_mode else selected_eeg()
        channels = set(cfg_entry.get('remap', {}).values())
        settings['mode'] = 'one_electrode' if one_mode else 'consensus'
        # only the electrodes this night has: an electrode ticked but absent would not change its scoring
        settings['EEG'] = ', '.join(sorted(e for e in wanted if e in channels)) or 'none'
        settings['EOG'] = dd_eog.value
    elif algo == 'YASA':
        settings['EEG'] = dd_yasa_eeg.value
        settings['EOG'] = dd_yasa_eog.value
        settings['chin EMG'] = dd_yasa_emg.value
    else:
        settings['tail context'] = 'whole night' if SLEEPFM_FULL_TAIL else f'{SFM.TAIL_CONTEXT_CHUNKS * 5} min'
    return settings


def skip_warning(algo, data_root, edf_path, fid):
    """The run-log warning of a scoring kept by Skip although made with other settings than this run's ('' when
    the settings are the same, or when its _scoring_params.json cannot be read)."""
    try:
        with open(output_paths(data_root, edf_path, fid, algo)['params'], encoding='utf-8') as f:
            stored = stored_scoring_settings(algo, json.load(f))
        changed = scoring_settings_mismatch(stored, run_settings(algo, edf_path, fid, STATE['cfg'][fid]))
    except Exception:
        return ''
    return f'⚠️ made with other settings (then → this run): {changed} (untick Skip to redo it)' if changed else ''


def score_model(algo, infer, edf_path, cfg_entry, sleepfm_root, progress=None):
    """One model on one night: a dict with at least stages, onsets, probs, reference, notes. `progress(done,
    total)`: GSSC's combinations, SleepFM's 5-min chunks (YASA predicts the night in one call)."""
    if algo == 'GSSC':
        one_mode = tb_mode.value.startswith('One')
        wanted = [dd_eeg.value] if one_mode else selected_eeg()
        r = score_night_gssc(infer, edf_path, cfg_entry, wanted, one_mode, dd_eog.value, dd_ref.value,
                             progress=progress)
        r['one_mode'] = one_mode
        return r
    if algo == 'YASA':
        return score_night_yasa(edf_path, cfg_entry, dd_yasa_eeg.value, dd_yasa_eog.value, dd_ref.value,
                                dd_yasa_emg.value)
    device = 'cuda' if (GSSC_OK and torch.cuda.is_available()) else 'cpu'
    r = score_night_sleepfm(edf_path, cfg_entry, sleepfm_root, dd_ref.value, device=device,
                            full_tail=SLEEPFM_FULL_TAIL, progress=progress)
    r['device'] = device
    return r


def score_one(algo, infer, data_root, fid, edf_path, cfg_entry, sleepfm_root=None, progress=None):
    """Score one participant with one model and write its outputs. Raises on a fatal error."""
    key = ALGO_KEY[algo]
    tag = f'[{fid}]' if algo == 'GSSC' else f'[{fid}] {algo}:'
    r = score_model(algo, infer, edf_path, cfg_entry, sleepfm_root, progress=progress)
    stages, onsets, probs, notes = r['stages'], r['onsets'], r['probs'], r['notes']
    for n in notes:
        log(f'{tag} ⚠️ {n}')
    for w in r.get('checks', []):
        log(f'{tag} ⚠️ channel check: {w}')

    # Expert hypnogram (optional). The models score 30-s epochs from the recording start and drop the
    # incomplete tail, like the scorer's epochs: epoch i = line i of the expert file.
    expert = None
    if hypno_suffix.value.strip():
        hp = expert_path(edf_path, fid)
        if hp is not None:
            expert = load_expert(hp)
            if abs(len(expert) - len(stages)) > 1:
                log(f'{tag} ⚠️ length mismatch: expert {len(expert)} epochs vs {algo} {len(stages)}: '
                    'compared on the common start')
        else:
            name = f'{fid}{hypno_suffix.value}'
            notes.append(f'no expert hypnogram ({name}): scored without comparison')
            log(f'{tag} no expert hypnogram ({name}): scored without comparison')

    start_dt = recording_start(edf_path)       # None if unreadable or the EDF+ placeholder 01.01.85 00.00.00
    if start_dt is None:
        notes.append('recording start time unknown (unreadable header, anonymised EDF placeholder, or no date in '
                     'the BrainVision markers or the Curry .dpo): no clock-time axis')
    paths = output_paths(data_root, edf_path, fid, algo)
    for d in {paths['density'].parent, paths['report'].parent}:
        d.mkdir(parents=True, exist_ok=True)

    # ---- per-participant data first (the report comes last: report on disk => data on disk) ----
    with open(paths['hypno'], 'w', encoding='utf-8') as f:
        f.write('\n'.join(stages) + '\n')
    dens = pd.DataFrame({'file_id': fid, 'epoch_idx': np.arange(len(stages)), 'onset_s': onsets,
                         f'stage_{key}': stages})
    for i, s in enumerate(STAGES):
        dens[f'p_{s}'] = probs[:, i]
    dens['confidence'] = probs.max(axis=1)
    if algo == 'GSSC':
        dens['winning_combination'] = r['winner']      # the signals whose scoring was kept for this epoch
    exp_col = np.full(len(stages), '', dtype=object)
    if expert is not None:
        n = min(len(expert), len(stages))
        exp_col[:n] = expert[:n]
    dens['stage_expert'] = exp_col
    # agree is left empty where the expert label is not W/N1/N2/N3/R (MT, custom, '?', not scored)
    dens['agree'] = np.where(~dens['stage_expert'].isin(STAGES), np.nan,
                             (dens['stage_expert'] == dens[f'stage_{key}']).astype(float))
    dens.to_csv(paths['density'], sep='\t', index=False, float_format='%.4f')
    if algo == 'SleepFM':
        # SleepFM's native 5-s probabilities (full precision: Section 4 averages them over its own epochs)
        # and its internal features + final layer (Section 4's recalibration B).
        five = pd.DataFrame({'file_id': fid, 'step_idx': np.arange(len(r['starts_s'])),
                             'start_s': r['starts_s'], 'stop_s': r['stops_s']})
        for i, s in enumerate(STAGES):
            five[f'p_{s}'] = r['probs_5s'][:, i]
        five['source'] = r['source']
        five.to_csv(paths['density_5s'], sep='\t', index=False)
        np.savez_compressed(paths['features'], features=r['features'], fc_weight=r['fc_weight'],
                            fc_bias=r['fc_bias'], start_s=r['starts_s'])

    common_end = {'expert_hypnogram_suffix': hypno_suffix.value if expert is not None else None,
                  'recording_start': start_dt.isoformat() if start_dt else None,
                  'notes': notes, 'scored_at': datetime.datetime.now().isoformat(timespec='seconds')}
    if algo == 'GSSC':
        eog_used = ', '.join(r['eog_names']) if r['eog_names'] else 'none'
        params = {'file_id': fid, 'scoring_mode': 'one_electrode' if r['one_mode'] else 'consensus',
                  'consensus_rule': CONSENSUS_RULE, 'n_combinations': r['n_comb'],
                  'hypnodensity': 'softmax of the winning combination (gssc 0.0.9 loop, logits kept)',
                  'eeg_used': r['present'], 'eog_used': r['eog_names'], 'eog_mode': dd_eog.value,
                  'reference': cfg_entry.get('ref_channels', []), 'eeg_reference': r['reference'],
                  'eog_declared': cfg_entry.get('context_channels') or {},
                  'expert_hypnogram_suffix': common_end['expert_hypnogram_suffix'],
                  'gssc_version': GSSC_VERSION, 'torch_version': TORCH_VERSION,
                  'device': 'cuda' if torch.cuda.is_available() else 'cpu',
                  'recording_start': common_end['recording_start'],
                  'notes': notes, 'scored_at': common_end['scored_at']}
        details = {'scoring_mode': params['scoring_mode'], 'eeg_used': ', '.join(r['present']),
                   'eog_used': eog_used}
        shown = {'EEG used': ', '.join(r['present']), 'EOG used': eog_used,
                 'Mode': f'{params["scoring_mode"]} ({r["n_comb"]} signal combinations, rule: most confident per epoch)',
                 'Reference (tool 2)': params['reference'], 'EEG reference (model)': r['reference'],
                 'GSSC / torch': f'{GSSC_VERSION} / {TORCH_VERSION}'}
    elif algo == 'YASA':
        used = r['used']
        params = {'file_id': fid, 'algorithm': 'YASA', 'eeg_used': used['eeg'], 'eog_used': used['eog'],
                  'emg_used': used['emg'], 'eog_choice': dd_yasa_eog.value, 'emg_choice': dd_yasa_emg.value,
                  'eeg_reference': r['reference'],
                  'reference': cfg_entry.get('ref_channels', []),
                  'context_declared': cfg_entry.get('context_channels') or {},
                  'demographics': 'not used (classifier without age/sex)', 'yasa_version': YASA_VERSION,
                  'antropy_version': ANTROPY_VERSION,
                  **common_end}
        details = {'eeg_used': used['eeg'], 'eog_used': used['eog'] or 'none', 'emg_used': used['emg'] or 'none'}
        shown = {'EEG used': used['eeg'], 'EOG used': used['eog'] or 'none', 'Chin EMG used': used['emg'] or 'none',
                 'EEG reference (model)': r['reference'], 'YASA': YASA_VERSION}
    else:
        used = r['used']
        params = {'file_id': fid, 'algorithm': 'SleepFM-Clinical', 'channels': used,
                  'eeg_reference': r['reference'], 'reference': cfg_entry.get('ref_channels', []),
                  'context_declared': cfg_entry.get('context_channels') or {},
                  'sleepfm_commit': SFM.SLEEPFM_COMMIT, 'sleepfm_folder': str(sleepfm_root),
                  'native_resolution_s': SFM.STEP_S,
                  'tail_context_min': None if r['tail_context_chunks'] is None else r['tail_context_chunks'] * 5,
                  'bandpass_hz': SFM.BANDPASS, 'device': r['device'], 'torch_version': TORCH_VERSION,
                  **common_end}
        details = {'eeg_used': ', '.join(used['BAS']), 'emg_used': ', '.join(used['EMG']) or 'none',
                   'resp_used': ', '.join(used['RESP']) or 'none'}
        shown = {'Brain channels (EEG + EOG)': ', '.join(used['BAS']), 'Chin EMG': ', '.join(used['EMG']) or 'none',
                 'ECG': ', '.join(used['EKG']) or 'none', 'Respiration': ', '.join(used['RESP']) or 'none',
                 'EEG reference (model)': r['reference'],
                 'SleepFM version': f'{SFM.SLEEPFM_COMMIT[:8]} (torch {TORCH_VERSION}, {r["device"]})'}
    # additive keys, absent when there is nothing to say (the file is then unchanged)
    if r.get('eog_reference'):
        params['eog_reference'] = r['eog_reference']
    if model_context_gain(algo, cfg_entry):
        params['context_gain'] = model_context_gain(algo, cfg_entry)
        shown['Context channels divided by (tool 2)'] = context_gain_text(params['context_gain'])
    if r.get('checks'):
        params['channel_quality_warnings'] = r['checks']
    with open(paths['params'], 'w', encoding='utf-8') as f:
        json.dump(params, f, indent=2, ensure_ascii=False, default=str)

    if expert is not None:
        res, cm, cm_long = compare_scoring(expert, stages, key)
        status = 'compared' if res['n_compared'] else 'no expert W/N1/N2/N3/R epoch'
    else:
        res, cm, cm_long = None, None, None
        status = 'no expert hypnogram'
    row = {'file_id': fid, f'n_epochs_{key}': len(stages),
           'n_epochs_expert': len(expert) if expert is not None else np.nan,
           'n_compared': res['n_compared'] if res else 0,
           'n_excluded_expert': res['n_excluded_expert'] if res else np.nan,
           'accuracy': res['accuracy'] if res else np.nan, 'kappa': res['kappa'] if res else np.nan,
           **{f'f1_{s}': (res[f'f1_{s}'] if res else np.nan) for s in STAGES},
           **{k: (res[k] if res else np.nan) for k in ('macro_f1', 'balanced_accuracy', 'mcc', 'n_stages_expert',
                                                        'single_stage_expert')},
           'mean_confidence': float(probs.max(axis=1).mean()),
           **details, 'status': status}
    pd.DataFrame([row]).to_csv(paths['comparison'], sep='\t', index=False, float_format='%.4f')
    # Expert epochs outside W/N1/N2/N3/R: what the model makes of them (only when the night has some).
    custom_path = paths['comparison'].parent / f'{fid}_custom_stages.tsv'
    profile = (non_aasm_profile(expert, stages, probs, STATE['custom'], key) if expert is not None
               else pd.DataFrame())
    if len(profile):
        profile.insert(0, 'file_id', fid)
        profile.to_csv(custom_path, sep='\t', index=False, float_format='%.4f')
    elif custom_path.exists():
        custom_path.unlink()                 # a former profile no longer valid
    if cm_long is not None:
        cm_long.insert(0, 'file_id', fid)
        cm_long.to_csv(paths['confusion'], sep='\t', index=False)
    elif paths['confusion'].exists():
        paths['confusion'].unlink()          # a former comparison no longer valid

    # ---- report ----
    if r.get('eog_reference'):
        shown['EOG reference (model)'] = r['eog_reference']
    shown.update({'Recording start': f'{start_dt:%Y-%m-%d %H:%M:%S}' if start_dt else 'unknown',
                  'Epochs scored': len(stages), 'Mean confidence': f'{row["mean_confidence"]:.2f}'})
    if notes:
        shown['Notes'] = '<br>'.join(f'⚠ {n}' for n in notes)
    if r.get('checks'):
        shown['Channel checks'] = ('<br>'.join(f'⚠ {w}' for w in r['checks'])
                                   + f'<br><small>{CHECKS_ADVICE}</small>')
    items = [('html', 'Scoring parameters', params_html(shown)),
             ('fig', 'Hypnograms', plot_hypnograms(stages, expert, fid, STATE['custom'], start_dt, algo)),
             ('fig', 'Hypnodensity', plot_hypnodensity(probs, fid, expert, start_dt))]
    if res is not None and res['n_compared']:
        items += [('html', 'Agreement with the expert', metrics_html(res)),
                  ('fig', 'Confusion matrix', plot_confusion(
                      cm, f'{fid}: kappa {res["kappa"]:.2f}, accuracy {res["accuracy"] * 100:.0f} %', algo))]
    else:
        items += [('html', 'Agreement with the expert',
                   f'<p>No comparison: {status}.</p>')]
    if len(profile):
        items += [('html', 'Expert stages outside W/N1/N2/N3/R', non_aasm_help(algo)),
                  ('fig', f'Expert stages outside W/N1/N2/N3/R: what {algo} makes of them',
                   plot_non_aasm_profile(profile, f'{fid}: expert stages outside W/N1/N2/N3/R, '
                                                  'with the AASM stages as reference', algo, key))]
    save_report_html(paths['report'], f'Automatic sleep scoring ({algo}): {fid}', items)
    return row


def night_start_hours(data_root, density_path, fid):
    """Recording start of a scored night on the noon-to-noon clock scale, or None: from its
    _scoring_params.json (recording_start), else from the recording's header (nights scored before that field)."""
    start_dt = None
    try:
        with open(density_path.parent / f'{fid}_scoring_params.json', encoding='utf-8') as f:
            iso = json.load(f).get('recording_start')
        start_dt = datetime.datetime.fromisoformat(iso) if iso else None
    except Exception:
        start_dt = None
    if start_dt is None:
        edf = STATE['edf_by_id'].get(fid) or next(
            (e for e in SF.list_recordings(data_root) if os.path.normcase(e.stem) == os.path.normcase(fid)), None)
        start_dt = recording_start(edf) if edf else None
    return clock_hours(start_dt) if start_dt else None


def rebuild_database_summary(data_root, algo='GSSC', single_stage_out=True):
    """One model's database tables + report, rebuilt from every per-participant table on disk.
    single_stage_out: the nights where the expert scored one stage only are left out of the kappa and the MCC
    summarised across nights (Section 1 option); the pooled values keep their epochs."""
    key = ALGO_KEY[algo]
    rep_root = data_root / f'reports_autoscoring_{key}'
    if not rep_root.exists():
        return
    # Per-participant tables only: the database tables written here (global_*) share their suffixes
    # (global_confusion.tsv, global_custom_stages.tsv) and would otherwise be counted again.
    def per_file(pattern):
        return sorted(p for p in rep_root.rglob(pattern) if not p.name.startswith('global_'))

    comps = per_file('*_autoscoring_comparison.tsv')
    if not comps:
        return
    summ = pd.concat([pd.read_csv(p, sep='\t', dtype={'file_id': str}) for p in comps], ignore_index=True)
    summ = summ.sort_values('file_id')
    summ.to_csv(rep_root / 'global_autoscoring_summary.tsv', sep='\t', index=False, float_format='%.4f')
    # nights where the expert scored one stage only (a column the nights compared before it do not have)
    single = (summ['single_stage_expert'].astype(str).str.lower() == 'true' if 'single_stage_expert' in summ
              else pd.Series(False, index=summ.index))
    confs = per_file('*_confusion.tsv')
    items = []
    pooled_path = rep_root / 'global_autoscoring_pooled.tsv'
    if confs:
        allc = pd.concat([pd.read_csv(p, sep='\t', dtype={'file_id': str}) for p in confs], ignore_index=True)
        pooled = allc.groupby(['expert', key], as_index=False)['n'].sum()
        pooled.to_csv(rep_root / 'global_confusion.tsv', sep='\t', index=False)
        cm = (pooled.pivot(index='expert', columns=key, values='n')
              .reindex(index=STAGES, columns=STAGES).fillna(0).astype(int).values)
        n_tot = cm.sum()
        acc = np.trace(cm) / n_tot if n_tot else np.nan
        # the metrics of every epoch pooled: the same function as a night's, on the summed confusion matrix
        res = agreement_metrics(cm)
        res['n_excluded_expert'] = int(pd.to_numeric(summ['n_excluded_expert'], errors='coerce').sum())
        pooled_cols = ['n_compared', 'n_excluded_expert', 'accuracy', 'kappa', 'mcc', 'macro_f1',
                       'balanced_accuracy', *[f'f1_{s}' for s in STAGES], 'n_stages_expert']
        pd.DataFrame([{'n_nights': allc['file_id'].nunique(), **{c: res[c] for c in pooled_cols}}]).to_csv(
            pooled_path, sep='\t', index=False, float_format='%.4f')
        items.append(('html', 'Agreement with the expert, every participant pooled', metrics_html(res)))
        items.append(('fig', 'Pooled confusion matrix', plot_confusion(
            cm, f'All participants pooled ({allc["file_id"].nunique()} nights, {n_tot} epochs), '
                f'accuracy {acc * 100:.0f} %', algo)))
    elif pooled_path.exists():
        pooled_path.unlink()                 # no comparison left: a former pooled table no longer valid
    customs = per_file('*_custom_stages.tsv')
    if customs:
        allp = pd.concat([pd.read_csv(p, sep='\t', dtype={'file_id': str}) for p in customs], ignore_index=True)
        pooled_p = pool_profiles(allp, STATE['custom'], key)
        pooled_p.to_csv(rep_root / 'global_custom_stages.tsv', sep='\t', index=False, float_format='%.4f')
        items.append(('html', 'Expert stages outside W/N1/N2/N3/R',
                      non_aasm_help(algo) + f'<p style="font-size:85%;color:#555">Pooled over the '
                      f'{allp["file_id"].nunique()} night(s) that have such epochs (reference bars from the '
                      'same nights).</p>'))
        items.append(('fig', f'Expert stages outside W/N1/N2/N3/R: what {algo} makes of them',
                      plot_non_aasm_profile(pooled_p, 'All nights pooled: expert stages outside W/N1/N2/N3/R',
                                            algo, key)))
    elif (rep_root / 'global_custom_stages.tsv').exists():
        (rep_root / 'global_custom_stages.tsv').unlink()
    comp = summ[summ['n_compared'] > 0]
    if len(comp):
        metrics = [(c, name) for c, name in [('kappa', "Cohen's kappa"), ('mcc', 'MCC'), ('accuracy', 'Accuracy'),
                                             ('balanced_accuracy', 'Balanced\naccuracy'), ('macro_f1', 'Macro-F1')]
                   if c in comp]
        fig, ax = plt.subplots(figsize=(9, 3.4))
        low = 0.0
        for i, (col, _) in enumerate(metrics):
            # kappa and MCC are 0 on a single-stage night whatever the model: left out when the option says so
            rows = comp[~single[comp.index]] if single_stage_out and col in ('kappa', 'mcc') else comp
            vals = pd.to_numeric(rows[col], errors='coerce').values
            ax.scatter(np.full(len(vals), i) + np.random.default_rng(0).uniform(-0.12, 0.12, len(vals)),
                       vals, s=18, color='#3a87c8', alpha=0.8)
            if np.isfinite(vals).any():
                ax.plot([i - 0.25, i + 0.25], [np.nanmedian(vals)] * 2, color='#222222', lw=2)
                low = min(low, np.nanmin(vals))
        ax.set_xticks(range(len(metrics)))
        ax.set_xticklabels([name for _, name in metrics])
        ax.set_ylim(low - 0.05 if low < 0 else 0, 1)          # kappa and MCC can be below 0
        ax.set_title('Agreement with the expert per participant (line = median)', fontsize=10)
        ax.grid(axis='y', alpha=0.3)
        fig.tight_layout()
        items.append(('fig', 'Agreement per participant', fig))
        notes = []
        n_single = int(single[comp.index].sum())
        if n_single:
            notes.append(f'{n_single} participant(s) where the expert scored a single stage: '
                         + ('left out of the kappa and the MCC (option of Section 1).' if single_stage_out else
                            'kept in the kappa and the MCC, where they count as 0 whatever the model (option of '
                            'Section 1).'))
        n_old = int(comp['macro_f1'].isna().sum()) if 'macro_f1' in comp else len(comp)
        if n_old:
            notes.append(f'{n_old} participant(s) compared before the MCC, the macro-F1 and the balanced accuracy '
                         'were added: these values are missing (–). Untick "Skip already processed participants" '
                         '(Section 1) to score them again, or compare them in Section 4, which computes every '
                         'metric from the saved scorings.')
        if notes:
            items.append(('html', 'About these values', ''.join(
                f'<p style="font-size:85%;color:#555">{n}</p>' for n in notes)))
    # Every participant's hypnodensity stacked on one time axis, for a quick visual comparison.
    dens_files = sorted((data_root / 'derivatives' / f'autoscoring_{key}').rglob('*_hypnodensity.tsv'),
                        key=lambda p: p.name)
    kappa_by_id = dict(zip(summ['file_id'].astype(str), pd.to_numeric(summ['kappa'], errors='coerce')))
    single_ids = set(summ.loc[single, 'file_id'].astype(str))
    nights = []
    for p in dens_files:
        try:
            d = pd.read_csv(p, sep='\t', dtype={'file_id': str}, keep_default_na=False)
            fid = str(d['file_id'].iloc[0])
            k = kappa_by_id.get(fid, np.nan)
            label = (f'{fid}\nκ –, 1 stage' if fid in single_ids
                     else fid if pd.isna(k) else f'{fid}\nκ {k:.2f}')
            expert = d['stage_expert'].astype(str).values if 'stage_expert' in d else None
            if expert is not None and not (expert != '').any():
                expert = None                                # scored without an expert hypnogram
            nights.append((label, d[[f'p_{s}' for s in STAGES]].astype(float).values, expert,
                           night_start_hours(data_root, p, fid)))
        except Exception as e:
            log(f'⚠️ Hypnodensity not shown in the database report ({p.name}): {e}')
    # Nights with a known start time on the real clock (one shared axis for every figure); the others
    # (start unknown) apart, on the relative axis.
    clocked = [n for n in nights if n[3] is not None]
    unclocked = [n for n in nights if n[3] is None]
    n_per = HYPNODENSITY_ROWS_PER_FIG
    if clocked:
        x_rng = (np.floor(min(n[3] for n in clocked)),
                 np.ceil(max(n[3] + len(n[1]) * EPOCH_SEC / 3600 for n in clocked)))
        for start in range(0, len(clocked), n_per):
            part = clocked[start:start + n_per]
            title = (f'Hypnodensities on clock time, participants {start + 1}-{start + len(part)} of '
                     f'{len(clocked)}' if len(clocked) > n_per
                     else f'Hypnodensities on clock time ({len(clocked)} participants)')
            items.append(('fig', title, plot_hypnodensity_stack(part, x_rng, title, clock=True)))
    if unclocked:
        x_rng = (0, max(len(n[1]) for n in unclocked) * EPOCH_SEC / 3600)
        for start in range(0, len(unclocked), n_per):
            part = unclocked[start:start + n_per]
            title = (f'Hypnodensities, recording start unknown ({len(unclocked)} '
                     f'participant(s), relative time)')
            items.append(('fig', title, plot_hypnodensity_stack(part, x_rng, title)))
    cols = ['file_id', f'n_epochs_{key}', 'n_compared', 'accuracy', 'kappa', 'mcc', 'macro_f1',
            'balanced_accuracy', 'single_stage_expert', 'mean_confidence',
            'scoring_mode', 'eeg_used', 'eog_used', 'emg_used', 'resp_used', 'status']
    items.append(('html', 'Participants', summ[[c for c in cols if c in summ]].to_html(
        index=False, float_format=lambda v: f'{v:.3f}', na_rep='–')))
    save_report_html(rep_root / 'global_autoscoring_report.html', f'Automatic sleep scoring ({algo}): database',
                     items)


def update_failed(data_root, algo, failed, attempted):
    """failed_autoscoring.tsv of one model: this run's failures + the earlier ones of participants not retried."""
    rep_root = data_root / f'reports_autoscoring_{ALGO_KEY[algo]}'
    fpath = rep_root / 'failed_autoscoring.tsv'
    old = pd.read_csv(fpath, sep='\t', dtype={'file_id': str}) if fpath.exists() else pd.DataFrame()
    if len(old):
        tried = {os.path.normcase(f) for f in attempted}
        old = old[~old['file_id'].map(os.path.normcase).isin(tried)]
    allf = pd.concat([old, pd.DataFrame(failed)], ignore_index=True)
    if len(allf):
        rep_root.mkdir(parents=True, exist_ok=True)
        allf.to_csv(fpath, sep='\t', index=False)
    elif fpath.exists():
        fpath.unlink()


def run_scoring(btn):
    btn.disabled = True
    out_run.clear_output()
    try:
        algos = ticked_algorithms()
        if not algos:
            log('❌ Tick at least one model (Section 2).')
            return
        if 'GSSC' in algos and not GSSC_OK:
            with out_run:
                display(HTML(GSSC_MSG))
            return
        if not fc_data.selected or not STATE['cfg']:
            log('Select the study folder and click Scan (Section 1) first.')
            return
        suffixes = {a: OUT_SUFFIX[a].value.strip() for a in algos}
        for a, suf in suffixes.items():
            if hypno_suffix.value.strip() and os.path.normcase(hypno_suffix.value.strip()) == os.path.normcase(suf):
                log(f'❌ The {a} output suffix equals the expert hypnogram suffix: the expert hypnograms would be '
                    'overwritten. Change the output suffix.')
                return
            if not suf.lower().endswith('.txt'):
                log(f'❌ The {a} output suffix must end with .txt.')
                return
        if len({os.path.normcase(s) for s in suffixes.values()}) < len(suffixes):
            log('❌ Two models have the same output suffix: give each its own.')
            return
        one_mode = tb_mode.value.startswith('One')
        wanted = [dd_eeg.value] if one_mode else selected_eeg()
        if 'GSSC' in algos and (not wanted or wanted == [None]):
            log('❌ Choose at least one EEG electrode for GSSC (Section 2).')
            return
        if 'YASA' in algos and not dd_yasa_eeg.value:
            log('❌ Choose the EEG electrode of YASA (Section 2).')
            return
        sleepfm_root = None
        if 'SleepFM' in algos:
            sleepfm_root = ensure_sleepfm(log)
            if sleepfm_root is None:
                log('⚠️ SleepFM is left out of this run (model unavailable); the other models still run.')
                algos = [a for a in algos if a != 'SleepFM']
                if not algos:
                    return
        data_root = Path(fc_data.selected)
        selected = participant_selector.selected_ids()
        if participant_selector.warning_html():
            with out_run:
                display(HTML(participant_selector.warning_html()))
        not_cfg = [f for f in selected if f in STATE['not_in_cfg']]
        if not_cfg:
            log(f'⚠️ Not in the tool-2 JSON, not scored: {", ".join(not_cfg)}')
        selected = [f for f in selected if f in STATE['cfg']]
        log(f'{len(selected)} participant(s) to score with {", ".join(algos)}; EEG reference: {dd_ref.value}.')
        if 'GSSC' in algos:
            log(f'GSSC: EEG {", ".join(wanted)}, EOG {dd_eog.value}.')
        if 'YASA' in algos:
            log(f'YASA: EEG {dd_yasa_eeg.value}, EOG {dd_yasa_eog.value}, chin EMG: {dd_yasa_emg.value}.')
        if 'SleepFM' in algos:
            # no choice for SleepFM: it takes every modality at once (the AASM electrodes present, both EOG, chin EMG,
            # ECG, respiration); one line per set of channels
            sleepfm_sets = {}
            for fid in selected:
                try:
                    text = sleepfm_channels_text(STATE['cfg'][fid], STATE.get('file_channels', {}).get(fid, ()))
                except Exception as e:
                    text = f'channels not listed ({e})'
                sleepfm_sets.setdefault(text, []).append(fid)
            for text, ids in sleepfm_sets.items():
                log(f'SleepFM: {text}' + (f' ({len(ids)} recording(s))' if len(sleepfm_sets) > 1 else '') + '.')
        selected_cfg = {f: STATE['cfg'][f] for f in selected}
        one_mode = tb_mode.value.startswith('One')
        for model, text in missing_channel_lines(selected_cfg, algos, gssc_eeg=wanted, gssc_one=one_mode,
                                                 gssc_eog=dd_eog.value, yasa_eeg=dd_yasa_eeg.value,
                                                 yasa_eog=dd_yasa_eog.value, yasa_emg=dd_yasa_emg.value,
                                                 file_channels=STATE.get('file_channels', {})):
            log(f'⚠️ {model}: {text}')

        # Skip ticked: a scoring already on disk is kept even when the settings changed since; a warning names
        # them. The participants every ticked model already scored are left out by the selector: checked here.
        n_other_settings = 0
        for fid in participant_selector.skipped_ids():
            if fid not in STATE['cfg']:
                continue
            for algo in algos:
                warning = skip_warning(algo, data_root, STATE['edf_by_id'][fid], fid)
                if warning:
                    n_other_settings += 1
                    log(f'[{fid}] {algo}: already scored, skipped.  {warning}')

        failed = {a: [] for a in algos}
        attempted = {a: set() for a in algos}
        progress.max = max(len(selected), 1)                   # one step per participant
        progress.value = 0
        progress.bar_style = 'info'
        progress_model.max = max(sum(MODEL_COST[a] for a in algos), 1)
        set_model_phases([(a, MODEL_COST[a], a) for a in algos])
        infer = None
        if selected and 'GSSC' in algos:
            try:
                infer = EEGInfer(use_cuda=torch.cuda.is_available())
            except Exception as e:
                log(f'❌ Cannot load the GSSC model: {e}')
                return
        for i, fid in enumerate(selected):
            edf = STATE['edf_by_id'][fid]
            progress_model.value = 0
            start = 0
            for algo in algos:
                base, cost = start, MODEL_COST[algo]
                start += cost
                # Skip ticked: a model whose outputs are complete is not run again (the others still are).
                if participant_selector.cb_skip.value and algo_outputs(data_root, edf, fid, algo)[0]:
                    warning = skip_warning(algo, data_root, edf, fid)
                    n_other_settings += bool(warning)
                    log(f'[{fid}] {algo}: already scored, skipped.' + (f'  {warning}' if warning else ''))
                    progress_model.value = start
                    continue
                attempted[algo].add(fid)
                t0 = time.time()
                lbl_model.value = f'<small>{fid}: {algo}</small>'
                unit = {'GSSC': 'combination', 'SleepFM': '5-min chunk'}.get(algo, 'step')

                def _model_progress(done, total, base=base, cost=cost, algo=algo, unit=unit):
                    progress_model.value = base + int(cost * done / max(total, 1))
                    lbl_model.value = f'<small>{fid}: {algo} ({unit} {done}/{total})</small>'

                try:
                    row = score_one(algo, infer, data_root, fid, edf, STATE['cfg'][fid], sleepfm_root,
                                    progress=_model_progress)
                    kappa = '' if pd.isna(row['kappa']) else f', kappa {row["kappa"]:.2f}'
                    log(f'[{fid}] ✅ {algo}: {row[f"n_epochs_{ALGO_KEY[algo]}"]} epochs scored{kappa} '
                        f'({time.time() - t0:.0f} s)')
                    if row['single_stage_expert'] is True:
                        log(f'[{fid}] ⚠️ {algo}: the expert scored a single stage on this night: its kappa and MCC '
                            'are 0 (or undefined) whatever the model; read its accuracy (see its report).')
                except Exception as e:
                    log(f'[{fid}] ❌ {algo}: {e}')
                    failed[algo].append({'file_id': fid, 'reason': str(e)})
                plt.close('all')
                progress_model.value = start
            progress.value += 1

        for algo in algos:
            try:
                update_failed(data_root, algo, failed[algo], attempted[algo])
            except Exception as e:
                log(f'⚠️ Could not update {algo} failed_autoscoring.tsv: {e}')
            try:
                rebuild_database_summary(data_root, algo, cb_single_stage_out.value)
                log(f'{algo} database summary rebuilt: {data_root / ("reports_autoscoring_" + ALGO_KEY[algo])}')
            except Exception as e:
                log(f'⚠️ Could not rebuild the {algo} database summary: {e}')
        # names registry (config_param/file_names.json): the suffix each model wrote, so that the hypnogram
        # pre-choice of the other tools sets these automatic hypnograms aside
        written = {f'hypnogram_{ALGO_KEY[a]}': suffixes[a] for a in algos if len(attempted[a]) > len(failed[a])}
        if written:
            names_warn = SF.write_names(data_root, '3bis', written)
            if names_warn:
                log(f'⚠️ {names_warn}')
        n_fail = sum(len(f) for f in failed.values())
        n_runs = sum(len(a) for a in attempted.values())
        UI.finish_progress(progress, n_fail)
        lbl_model.value = ''
        log(f'Done: {n_runs - n_fail} scoring(s) done, {n_fail} failed.')
        if n_other_settings:
            log(f'⚠️ {n_other_settings} scoring(s) kept although made with other settings (the ⚠️ lines above): '
                'untick "Skip already processed participants" (Section 1) to redo them with this run\'s settings.')
        refresh_done()
    except Exception as e:
        log(f'❌ Unexpected error: {e}')
    finally:
        UI.stop_working()
        btn.disabled = False


btn_run.on_click(run_scoring)
display(btn_run, progress_row, progress_model_row, out_run)

<hr style="height:4px; background-color:black; border:none;">

## 4. Compare the models

Compares the models scored in Section 3 with the expert hypnogram, on every participant chosen in Section 1
(the *Skip* option does not apply here) that has an expert hypnogram. Only the files written by Section 3 are
read: nothing is scored again. Click **Find the scorings**, choose the models to compare, then **Compare**.

How it works (the computation of Score_EDF): 30-s epochs from the first to the last scored expert epoch; each
model's probabilities averaged over each epoch; the predicted stage is the most probable one. SleepFM's two
**recalibrations** are learnt on the expert hypnograms of the other nights (leave one night out): A needs at
least 2 nights with SleepFM, B at least 3 (6 or more advised). An epoch a model gave no stage for (the last
epoch GSSC drops, see Details) is left out of its metrics and counted.

**A comparison is one set of nights.** The recalibrations learn each night from the others, so a night's results
belong to the nights compared with it: each **Compare** recomputes every chosen night and **replaces the previous
comparison** (its two folders are emptied first; *Skip* does not apply).

**Outputs**: per night, `derivatives/autoscoring_comparison/<subtree>/{id}_comparison_epochs.tsv` (one row per
epoch and model: expert and model stages, probabilities) and, in `reports_autoscoring_comparison/<subtree>/`,
`{id}_comparison.tsv` (the agreement metrics and the channels used, one row per model), `{id}_confusion.tsv` and
`{id}_comparison_report.html` (the hypnodensities of every model over the expert hypnogram). For the database,
in `reports_autoscoring_comparison/`: `global_comparison_summary.tsv`, `global_comparison_pooled.tsv` (every night
pooled), `global_confusion.tsv` and `global_comparison_report.html`; SleepFM's recalibration parameters in
`derivatives/autoscoring_comparison/recalibration_params.tsv`. Each model is given the channels it was trained on,
so two models may not see the same EEG: to compare them on equal inputs, give them the same electrode in Section 2.

**How to read the results**
- **Agreement table**: per model, the number of epochs compared, the **accuracy** (share of epochs where the
  model gives the expert's stage), **Cohen's kappa** (agreement corrected for chance: 0.61-0.80 good, above
  0.80 excellent; two human scorers reach about 0.76, i.e. ~83 % of identical epochs), the **MCC** (correlation
  between the expert and the model, 0 = chance), the **macro-F1** (mean of the F1 over the stages the expert
  scored), the **balanced accuracy** and the **F1 per stage** (balance between finding every epoch of the stage
  and not inventing any; N1 is always the lowest, between humans too). A night where the expert scored a
  **single stage** has a kappa and an MCC of 0 whatever the model: the option of Section 1 leaves it out of the
  kappa of each night below (the pooled values keep its epochs).
- **Agreement figure**: left, F1 per stage and model; right, the kappa of each night (one symbol per night, the
  black dash = all nights pooled). A night far below the others for every model points to a signal or scoring
  problem rather than to a model.
- **Confusion matrices**: rows = expert stage, columns = predicted stage, % of the row. The diagonal = agreement.
  Usual confusions: N1 with wake, N1 with N2, N2 with N3.
- **Hypnogram grid**: one band per night, the expert on top then one line per model, one colour per stage (grey
  = no data): where the models depart from the expert across the night, at a glance.
- **Hypnodensities** (in each night's report): each model's probability of each stage across the night (stacked
  areas), the expert hypnogram below. One dominant colour = a confident model; mixed colours = an ambiguous epoch
  (often transitions and N1). They show the confidence, not only the stage.

In [ ]:
cmp_checks = {name: widgets.Checkbox(value=False, description=name, disabled=True, indent=False,
                                     layout=widgets.Layout(width='420px')) for name in ACL.STAGERS}
btn_compare_find = UI.button('Find the scorings', 'read')
cmp_info = widgets.HTML('')
btn_compare = UI.button('Compare the models', 'run')
out_compare = widgets.Output()
out_results = widgets.Output()
# Expert switch (no widget): False reproduces Score_EDF's recalibrations exactly (N3 never fitted); 'auto'
# fits N3 when the training nights have N3 (plan 2.7).
RECAL_FREE_N3 = 'auto'
COMPARE = {}


def clog(msg):
    with out_compare:
        print(msg)


def step1_ready(data_root, edf, fid, algo):
    p = output_paths(data_root, edf, fid, algo)
    return p['density'].exists() and (algo != 'SleepFM' or p['density_5s'].exists())


def step1_start(data_root, edf, fid):
    """Recording start read from step 1's parameter files (no EDF read), else None."""
    for algo in ALGORITHMS:
        p = output_paths(data_root, edf, fid, algo)['params']
        try:
            iso = json.loads(p.read_text(encoding='utf-8')).get('recording_start')
            if iso:
                return datetime.datetime.fromisoformat(iso)
        except Exception:
            continue
    return None


def compare_candidates(data_root):
    """Participants chosen in Section 1 that have an expert hypnogram, with the models scored for them."""
    rows = {}
    for fid in participant_selector.chosen_ids():
        edf = STATE['edf_by_id'].get(fid)
        if edf is None:
            continue
        hp = expert_path(edf, fid)
        if hp is None:
            continue
        rows[fid] = {'edf': edf, 'expert': hp,
                     'algos': [a for a in ALGORITHMS if step1_ready(data_root, edf, fid, a)]}
    return rows


def on_compare_find(btn):
    btn.disabled = True
    try:
        if not fc_data.selected or not STATE['edf_by_id']:
            cmp_info.value = f'<span style="color:{UI.ERROR_COLOR};">Scan the study folder first (Section 1).</span>'
            return
        data_root = Path(fc_data.selected)
        cands = compare_candidates(data_root)
        n = {a: sum(a in c['algos'] for c in cands.values()) for a in ALGORITHMS}
        n_feat = sum('SleepFM' in c['algos'] and output_paths(data_root, c['edf'], fid, 'SleepFM')['features'].exists()
                     for fid, c in cands.items())
        notes = {a: (f' ({n[a]} night(s))' if n[a] else ' (not scored in Section 3)') for a in ALGORITHMS}
        enabled = {a: n[a] > 0 for a in ALGORITHMS}
        enabled['SleepFM recal A'] = n['SleepFM'] >= 2
        notes['SleepFM recal A'] = (f' (learnt on the other nights: {n["SleepFM"]} with SleepFM)' if n['SleepFM'] >= 2
                                    else f' (needs >= 2 nights with SleepFM, have {n["SleepFM"]})')
        enabled['SleepFM recal B'] = n_feat >= 3
        notes['SleepFM recal B'] = (f' ({n_feat} nights; works best with >= 6)' if 3 <= n_feat < 6
                                    else (f' ({n_feat} nights)' if n_feat >= 6
                                          else f' (needs >= 3 nights with SleepFM, have {n_feat})'))
        for name, cb in cmp_checks.items():
            cb.disabled = not enabled[name]
            cb.value = enabled[name]
            cb.description = name + notes[name]
        cmp_info.value = (f'<span style="color:{UI.OK_COLOR};">{len(cands)} participant(s) with an expert hypnogram '
                          f'({hypno_suffix.value}).</span>' if cands else
                          f'<span style="color:{UI.ERROR_COLOR};">No participant with an expert hypnogram (check the expert '
                          'suffix in Section 1).</span>')
        COMPARE['candidates'] = cands
    except Exception as e:
        cmp_info.value = f'<span style="color:{UI.ERROR_COLOR};">Error: {e}</span>'
    finally:
        btn.disabled = False


def channels_used(data_root, cands, selected):
    """One row per night and model of step 1: the channels and the EEG reference the model used, read from step 1's
    _scoring_params.json (no EDF read). The recalibrated SleepFM uses SleepFM's."""
    models = [a for a in ALGORITHMS if a in selected or (a == 'SleepFM' and any(s.startswith('SleepFM')
                                                                                 for s in selected))]
    rows = []
    for fid, c in cands.items():
        for algo in models:
            if algo not in c['algos']:
                continue
            try:
                d = json.loads(output_paths(data_root, c['edf'], fid, algo)['params'].read_text(encoding='utf-8'))
            except Exception:
                continue
            if algo == 'GSSC':
                # GSSC records its EOG as EOG-L / EOG-R / EOG L-R: shown with the channels declared in tool 2
                decl = d.get('eog_declared') or {}
                names = {'EOG-L': decl.get('eog_left'), 'EOG-R': decl.get('eog_right'),
                         'EOG L-R': f"{decl.get('eog_left')} - {decl.get('eog_right')}"}
                eog = ', '.join(names.get(e) or e for e in (d.get('eog_used') or [])) or 'none'
                row = {'eeg_used': ', '.join(d.get('eeg_used') or []), 'eog_used': eog, 'emg_used': '-',
                       'other_channels': f"{d.get('scoring_mode')} ({d.get('n_combinations')} combinations)"}
            elif algo == 'YASA':
                row = {'eeg_used': d.get('eeg_used'), 'eog_used': d.get('eog_used') or 'none',
                       'emg_used': d.get('emg_used') or 'none', 'other_channels': ''}
            else:
                ch = d.get('channels') or {}
                eeg = [n for n in ch.get('BAS', []) if not n.startswith('EOG')]
                row = {'eeg_used': ', '.join(eeg), 'eog_used': ', '.join(ch.get('EOG', [])) or 'none',
                       'emg_used': ', '.join(ch.get('EMG', [])) or 'none',
                       'other_channels': f"ECG: {', '.join(ch.get('EKG', [])) or 'none'}; respiration: "
                                         f"{', '.join(ch.get('RESP', [])) or 'none'}"}
            rows.append({'file_id': fid, 'model': ACL.MODEL_KEY[algo], **row,
                         'eeg_reference': d.get('eeg_reference', '')})
    return pd.DataFrame(rows)


def comparison_dirs(data_root, edf):
    """Section 4's data and report folders of a night (the subtree of its recording, as in step 1)."""
    rel = edf.parent.relative_to(data_root)
    return (data_root / 'derivatives' / 'autoscoring_comparison' / rel,
            data_root / 'reports_autoscoring_comparison' / rel)


def fig_png(fig):
    """A figure as PNG bytes, to show in the page: this tool draws without a screen (Agg backend, cell 1), so
    display(fig) would only print '<Figure size ...>' (tools 0, 8 and 8bis show a PNG too)."""
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=110, bbox_inches='tight')
    return buf.getvalue()


def agreement_html(table):
    """An agreement table (one row per model, one night or every night pooled) with the models' display names."""
    cols = ['model', 'n_nights', 'n_compared', 'n_not_predicted', 'accuracy', 'kappa', 'mcc', 'macro_f1',
            'balanced_accuracy', *[f'f1_{s}' for s in STAGES]]
    shown = table[[c for c in cols if c in table]].copy()
    shown['model'] = shown['model'].map(ACL.MODEL_NAME)
    return shown.to_html(index=False, float_format=lambda v: f'{v:.3f}', na_rep='–')


SINGLE_STAGE_NOTE = ('<p style="font-size:85%;color:#e65100">⚠ The expert scored a single stage on this night: the '
                     'kappa and the MCC are 0 (or undefined) whatever the model, by construction. Read the accuracy, '
                     'which here also gives the balanced accuracy and the macro-F1.</p>')


def save_night_report(reports, fid, epochs, table, confusion):
    """One night's report: the agreement of each model, their hypnodensities over the expert hypnogram, their
    confusion matrices and the channels each one used."""
    note = SINGLE_STAGE_NOTE if table['single_stage_expert'].any() else ''
    channel_cols = ['model', 'eeg_used', 'eog_used', 'emg_used', 'other_channels', 'eeg_reference']
    channels = table[[c for c in channel_cols if c in table]].copy()
    channels['model'] = channels['model'].map(ACL.MODEL_NAME)
    items = [('html', 'Agreement with the expert', agreement_html(table) + note),
             ('fig', 'Hypnodensities', ACL.plot_hypnodensities(epochs, fid)),
             ('fig', 'Confusion matrices', ACL.plot_confusion_matrices(
                 confusion, f'{fid}: confusion matrices, % of the expert stage')),
             ('html', 'Channels used by each model', channels.to_html(index=False, na_rep='–'))]
    save_report_html(reports / f'{fid}_comparison_report.html', f'Comparison of the automatic scorings: {fid}', items)


def rebuild_comparison_database(data_root, single_stage_out=True):
    """Section 4's database tables and report, rebuilt from the per-night files on disk (they are the last comparison:
    each Compare empties both folders first). single_stage_out: the Section 1 option. Returns what the page shows,
    a list of ('heading' | 'html' | 'png', title, payload)."""
    deriv_root = data_root / 'derivatives' / 'autoscoring_comparison'
    rep_root = data_root / 'reports_autoscoring_comparison'

    def per_file(root, pattern):
        return sorted(p for p in root.rglob(pattern) if not p.name.startswith('global_'))

    tables = per_file(rep_root, '*_comparison.tsv')
    if not tables:
        return []
    summary = pd.concat([pd.read_csv(p, sep='\t', dtype={'file_id': str}) for p in tables], ignore_index=True)
    summary.to_csv(rep_root / 'global_comparison_summary.tsv', sep='\t', index=False, float_format='%.4f')
    confusion = pd.concat([pd.read_csv(p, sep='\t', dtype={'file_id': str}) for p in per_file(rep_root, '*_confusion.tsv')],
                          ignore_index=True)
    (confusion.groupby(['model', 'expert', 'predicted'], sort=False, as_index=False)['n'].sum()
     .to_csv(rep_root / 'global_confusion.tsv', sep='\t', index=False))
    pooled = ACL.pooled_agreement(confusion, summary)
    pooled.to_csv(rep_root / 'global_comparison_pooled.tsv', sep='\t', index=False, float_format='%.4f')
    epochs = pd.concat([pd.read_csv(p, sep='\t', dtype={'file_id': str, 'stage_expert': str, 'stage_model': str})
                        for p in per_file(deriv_root, '*_comparison_epochs.tsv')], ignore_index=True)
    epochs[['stage_expert', 'stage_model']] = epochs[['stage_expert', 'stage_model']].fillna('')

    n_single = summary.loc[ACL.is_true(summary['single_stage_expert']), 'file_id'].nunique()
    note = ''
    if n_single:
        note = (f'<p style="font-size:85%;color:#555">{n_single} night(s) where the expert scored a single stage: '
                + ('left out of the kappa of each night (option of Section 1); the pooled values keep their epochs.'
                   if single_stage_out else 'kept in the kappa of each night, where they count as 0 whatever the '
                   'model (option of Section 1).') + '</p>')
    figures = [('Agreement per stage and per night', ACL.plot_agreement_summary(pooled, summary, single_stage_out)),
               ('Confusion matrices', ACL.plot_confusion_matrices(
                   confusion, 'Confusion matrices, % of the expert stage (every night pooled)'))]
    file_ids = sorted(epochs['file_id'].unique())
    n_per = HYPNODENSITY_ROWS_PER_FIG
    for start in range(0, len(file_ids), n_per):
        part = file_ids[start:start + n_per]
        title = 'Hypnogram comparison' + (f', nights {start + 1}-{start + len(part)} of {len(file_ids)}'
                                          if len(file_ids) > n_per else '')
        figures.append((title, ACL.plot_hypnogram_grid(epochs, part)))
    # the page shows the PNG of each figure: taken before the report saves (and closes) them
    pngs = {title: fig_png(fig) for title, fig in figures}
    pooled_html = agreement_html(pooled) + note
    items = [('html', 'Agreement with the expert, every night pooled', pooled_html)]
    items += [('fig', title, fig) for title, fig in figures]
    per_night = summary.copy()
    per_night['model'] = per_night['model'].map(ACL.MODEL_NAME)
    items.append(('html', 'Per night', per_night.to_html(index=False, float_format=lambda v: f'{v:.3f}', na_rep='–')))
    recal = deriv_root / 'recalibration_params.tsv'
    if recal.exists():
        items.append(('html', 'Recalibration parameters of SleepFM (leave one night out)', pd.read_csv(
            recal, sep='\t', dtype={'file_id': str}).to_html(index=False, float_format=lambda v: f'{v:.4g}', na_rep='–')))
    save_report_html(rep_root / 'global_comparison_report.html', 'Comparison of the automatic scorings: database', items)

    page = [('heading', '4.1', 'Agreement with the expert'), ('html', 'Every night pooled', pooled_html),
            ('png', 'Agreement per stage and per night', pngs['Agreement per stage and per night']),
            ('heading', '4.2', 'Confusion matrices'), ('png', '', pngs['Confusion matrices']),
            ('heading', '4.3', 'Hypnogram comparison')]
    page += [('png', '', png) for title, png in pngs.items() if title.startswith('Hypnogram comparison')]
    return page


def run_compare(btn):
    btn.disabled = True
    out_compare.clear_output()
    out_results.clear_output()
    try:
        data_root = Path(fc_data.selected) if fc_data.selected else None
        cands = COMPARE.get('candidates')
        selected = [n for n, cb in cmp_checks.items() if cb.value and not cb.disabled]
        if data_root is None or not cands:
            clog('Click "Find the scorings" first.')
            return
        if not selected:
            clog('⛔ Tick at least one model to compare.')
            return
        deriv_root = data_root / 'derivatives' / 'autoscoring_comparison'
        rep_root = data_root / 'reports_autoscoring_comparison'
        clog(f'Models compared: {selected}')
        clog(f'Recordings: {len(cands)}')
        if (data_root / 'autoscore_outputs').is_dir():
            clog('Note: autoscore_outputs/ holds a former comparison (the layout of Score_EDF): it is no longer '
                 'written and is left as it is.')
        needs_sleepfm = any(s.startswith('SleepFM') for s in selected)
        five = {name: {} for name in selected}
        native = {}
        info = {}
        failed = []
        for fid, c in cands.items():
            try:
                hyp = ACL.read_expert(c['expert'])
                window = ACL.window_from_expert(hyp, step1_start(data_root, c['edf'], fid))
                clog(f'--- {fid} ---  expert: {c["expert"].name} ({len(hyp)} epochs), window '
                     f'{window.lights_off_clock} -- {window.reveil_clock} ({window.duration_seconds / 60:.1f} min)')
                info[fid] = {'window': window, 'hypnogram': hyp}
                for algo in ('YASA', 'GSSC', 'SleepFM'):
                    if algo not in c['algos'] or not (algo in selected or (algo == 'SleepFM' and needs_sleepfm)):
                        continue
                    p = output_paths(data_root, c['edf'], fid, algo)
                    frame, nat = ACL.step1_frame(p['density'], p['density_5s'] if algo == 'SleepFM' else None)
                    if algo in selected:
                        five[algo][fid] = frame
                        native[(algo, fid)] = nat
                    if algo == 'SleepFM':
                        info[fid]['sleepfm_5s'] = frame
                        info[fid]['features'] = p['features']
            except Exception as e:
                clog(f'[{fid}] ❌ {e}')
                failed.append((fid, str(e)))

        # ---- SleepFM recalibrations, leave one recording out ----
        recal_rows, recal_layers = [], {}
        recal_subjects = [fid for fid in info if 'sleepfm_5s' in info[fid]]
        needs_recal = any(s in selected for s in ('SleepFM recal A', 'SleepFM recal B'))
        if needs_recal and len(recal_subjects) >= 2:
            base_recs = [ACL.prepare_recording(fid, info[fid]['sleepfm_5s'], info[fid]['window'],
                                               info[fid]['hypnogram']) for fid in recal_subjects]
            if 'SleepFM recal A' in selected:
                clog('\n--- Recalibration A (leave one recording out) ---')
                five['SleepFM recal A'], rows = ACL.recalibrate_a(base_recs, RECAL_FREE_N3, clog)
                recal_rows += rows
            if 'SleepFM recal B' in selected and len(base_recs) >= 3:
                clog('\n--- Recalibration B (leave one recording out, refit of the final layer) ---')
                feats, fc_w, fc_b = {}, None, None
                for fid in recal_subjects:
                    d = np.load(info[fid]['features'])
                    feats[fid], fc_w, fc_b = d['features'], d['fc_weight'], d['fc_bias']
                five['SleepFM recal B'], rows, recal_layers = ACL.recalibrate_b(base_recs, feats, fc_w, fc_b,
                                                                                RECAL_FREE_N3, clog)
                recal_rows += rows

        active = [s for s in selected if five.get(s)]
        if not active:
            clog('No results to display.')
            return
        # A comparison is one set of nights (the recalibrations learn each night from the others): each Compare
        # replaces the previous one, so the database tables, rebuilt from the files on disk, are this comparison.
        for folder in (deriv_root, rep_root):
            if folder.exists():
                shutil.rmtree(folder)
        clog(f'\nOutputs (both folders emptied first): {deriv_root} and {rep_root}')
        channels = channels_used(data_root, cands, selected)
        nights = []
        for fid in sorted(info):
            try:
                frames = [ACL.toolkit_epochs(ACL.build_epoch_table(info[fid]['window'], info[fid]['hypnogram'],
                                                                   five[name][fid], 30.0,
                                                                   native_stages=native.get((name, fid))), fid, name)
                          for name in active if fid in five[name]]
                if not frames:
                    continue
                epochs = pd.concat(frames, ignore_index=True)
                deriv, reports = comparison_dirs(data_root, cands[fid]['edf'])
                deriv.mkdir(parents=True, exist_ok=True)
                reports.mkdir(parents=True, exist_ok=True)
                epochs.to_csv(deriv / f'{fid}_comparison_epochs.tsv', sep='\t', index=False)
                if fid in recal_layers:
                    weight, bias = recal_layers[fid]
                    np.savez(deriv / f'{fid}_recal_b_layer.npz', weight=weight, bias=bias)
                table, confusion = ACL.night_agreement(epochs)
                night_channels = channels[channels['file_id'] == fid] if len(channels) else channels
                if len(night_channels):
                    # the recalibrated SleepFM used SleepFM's channels
                    base = table['model'].str.replace(r'_recal_[ab]$', '', regex=True)
                    table = (table.assign(base=base)
                             .merge(night_channels.drop(columns='file_id').rename(columns={'model': 'base'}),
                                    on='base', how='left').drop(columns='base'))
                table.insert(0, 'file_id', fid)
                table.to_csv(reports / f'{fid}_comparison.tsv', sep='\t', index=False, float_format='%.4f')
                confusion.insert(0, 'file_id', fid)
                confusion.to_csv(reports / f'{fid}_confusion.tsv', sep='\t', index=False)
                nights.append((fid, epochs, table, confusion, reports))
                if table['single_stage_expert'].any():
                    clog(f'[{fid}] ⚠️ the expert scored a single stage on this night: its kappa and MCC are 0 (or '
                         'undefined) whatever the model.')
            except Exception as e:
                clog(f'[{fid}] ❌ {e}')
                failed.append((fid, str(e)))
        if recal_rows:
            deriv_root.mkdir(parents=True, exist_ok=True)
            pd.DataFrame(recal_rows).to_csv(deriv_root / 'recalibration_params.tsv', sep='\t', index=False)
        for fid, epochs, table, confusion, reports in nights:       # the reports once every data file is written
            try:
                save_night_report(reports, fid, epochs, table, confusion)
            except Exception as e:
                clog(f'[{fid}] ⚠️ report not written: {e}')
            plt.close('all')
        clog(f'{len(nights)} night(s) written.')
        if failed:
            rep_root.mkdir(parents=True, exist_ok=True)
            pd.DataFrame(failed, columns=['file_id', 'reason']).to_csv(rep_root / 'failed_comparison.tsv', sep='\t',
                                                                       index=False)
            clog(f'\nFailed recordings ({len(failed)}), listed in failed_comparison.tsv:')
            for fid, err in failed:
                clog(f'  {fid}: {err}')
        try:
            page = rebuild_comparison_database(data_root, cb_single_stage_out.value)
        except Exception as e:
            clog(f'⚠️ Could not rebuild the database tables and report: {e}')
            page = []
        show_comparison(page, rep_root)
        clog('Done.')
    except Exception as e:
        clog(f'❌ Unexpected error: {e}')
    finally:
        btn.disabled = False


def show_comparison(page, rep_root):
    """The database view of the comparison in the page; each night's hypnodensities are in its report."""
    with out_results:
        for kind, title, payload in page:
            if kind == 'heading':
                display(HTML(UI.subsection_html(title, payload)))
            elif kind == 'html':
                display(HTML(f'<h4>{title}</h4>{payload}'))
            else:
                if title:
                    display(HTML(f'<h4>{title}</h4>'))
                display(widgets.Image(value=payload, format='png', layout=widgets.Layout(max_width='100%', height='auto')))
        display(HTML(UI.msg('ok', f'Reports in <code>{rep_root}</code>: one per night (with the hypnodensities of every '
                                  'model) and <code>global_comparison_report.html</code>.')))


btn_compare_find.on_click(on_compare_find)
btn_compare.on_click(run_compare)
display(btn_compare_find, cmp_info, widgets.HTML('<b>Models to compare:</b>'),
        widgets.VBox(list(cmp_checks.values())), btn_compare, out_compare, out_results)

<hr style="height:4px; background-color:black; border:none;">

## References

- Vallat R, Walker MP. An open-source, high-performance tool for automated sleep staging. eLife. 2021;10:e70092.
  doi:[10.7554/eLife.70092](https://doi.org/10.7554/eLife.70092) (YASA)
- Hanna J, Flöel A. An accessible and versatile deep learning-based sleep stage classifier. Front Neuroinform.
  2023;17:1086634. doi:[10.3389/fninf.2023.1086634](https://doi.org/10.3389/fninf.2023.1086634) (GSSC)
- Thapa R, et al. A multimodal sleep foundation model for disease prediction. Nat Med. 2026.
  doi:[10.1038/s41591-025-04133-4](https://doi.org/10.1038/s41591-025-04133-4) (SleepFM). Code and model:
  https://github.com/zou-group/sleepfm-clinical
- Troester MM, Quan SF, Berry RB, et al. The AASM manual for the scoring of sleep and associated events: rules,
  terminology and technical specifications. Version 3. Darien (IL): American Academy of Sleep Medicine; 2023.
- Cohen J. A coefficient of agreement for nominal scales. Educ Psychol Meas. 1960;20(1):37-46.
  doi:[10.1177/001316446002000104](https://doi.org/10.1177/001316446002000104) (Cohen's kappa)
- Landis JR, Koch GG. The measurement of observer agreement for categorical data. Biometrics. 1977;33(1):159-74.
  doi:[10.2307/2529310](https://doi.org/10.2307/2529310) (the kappa bands: 0.61-0.80 good, above 0.80 excellent)
- Rosenberg RS, Van Hout S. The American Academy of Sleep Medicine inter-scorer reliability program: sleep stage
  scoring. J Clin Sleep Med. 2013;9(1):81-7. doi:[10.5664/jcsm.2350](https://doi.org/10.5664/jcsm.2350) (the
  agreement between two human scorers)